# Are friend communities also taste communities? — full pipeline

DTU 02807 Computational Tools for Data Science, Yelp Open Dataset.

This notebook puts the five step notebooks together, in their original order. Each step starts with its own setup cell and reads the files written by the steps before it, so the notebook runs top to bottom with *Run All* (raw Yelp files in `data/raw/`; install `requirements.txt` first).

| Step | Section | Source notebook |
|---|---|---|
| 1 | Step 1 — Data cleaning & preprocessing | `01_data_cleaning.ipynb` |
| 2 | Step 2 — Friend communities (Louvain, implemented from scratch) | `02_friend_communities.ipynb` |
| 3 | Step 3 — Taste communities | `03_taste_communities.ipynb` |
| 4 | Step 4 — Frequent itemsets and association rules (A-priori vs. PCY) | `04_frequent_itemsets.ipynb` |
| 5 | Step 5 — Comparison of friend and taste communities | `05_comparison.ipynb` |


---
*Step 1 — from `01_data_cleaning.ipynb`.*

# Step 1 — Data cleaning & preprocessing

**Project question:** *Are friend communities also taste communities?*
(DTU 02807 Computational Tools for Data Science, Yelp Open Dataset)

*Code: all in this notebook (no `src/` module); it runs the cleaning and explains the choices.*

**Question.** Which users, friendships and reviews do the later steps work on? Here we turn the raw Yelp files into clean tables for one city, and record every step, the reason for it and what we found, so everyone in the group (and the graders) can follow and reproduce the cleaning.

**Plan**
- **1a** Load and check the **business** file; normalise city names; parse `categories`; flag **restaurants** (our scope).
- **1b** Stream the 5.3 GB **review** file once and keep only the columns we need (cached as Parquet).
- **1c** **Choose a single city**, using evidence: how many active restaurant reviewers and how large a friend graph each city gives.
- **1d** **Filter**: city → restaurants → one review per (user, restaurant) → users with ≥ 10 restaurant reviews → largest connected component (LCC) of the friendship graph. Row counts are logged after every filter.
- **1e** **Normalise star ratings per user** (subtract the user's mean) and mark **positive ratings** (stars ≥ 3.5).
- **1f** Quick exploration of the cleaned data.
- **1g** Sanity checks, then **save the clean tables** to `data/processed/` for steps 2–5.

**Outputs:** four Parquet files (users, friend edges, reviews, restaurants) plus a JSON summary of the cleaning. They are described in section 1g.

## Before you run: setup (5 minutes + download)

**1. Get the data.** Download the **Yelp Open Dataset (JSON)** from <https://business.yelp.com/data/resources/open-dataset/> and accept its terms. We cannot share the data ourselves. Extract the archive. Only **3 of its 5 files** are needed:

- `yelp_academic_dataset_business.json`
- `yelp_academic_dataset_review.json`
- `yelp_academic_dataset_user.json`

Tips and check-ins are not used.

**2. Put the files here.** Create this folder layout. Only `data/raw/` needs to be created by hand: `data/processed/` and `figures/` are created by the notebook.
```
project/
├── notebooks/
│   └── 01_data_cleaning.ipynb      <- this notebook
└── data/
    └── raw/                        <- the 3 JSON files above (≈ 8.8 GB)
```
If the files are already somewhere else, you can leave them there and set the environment variable `YELP_DATA_RAW` to that folder before starting Jupyter, e.g. `export YELP_DATA_RAW="/path/to/yelp_dataset"`.

**3. Install the packages** (Python ≥ 3.10):
```bash
pip install pandas numpy pyarrow networkx matplotlib jupyterlab
```

**4. Run.** `jupyter lab notebooks/01_data_cleaning.ipynb`, then *Run → Run All Cells*.
- The first run streams the raw files and takes about **2 minutes**. It writes caches to `data/processed/`.
- After that, a re-run takes under 30 seconds.
- The large raw files are never loaded whole, so a laptop with 8 GB of RAM is enough.
- If the data folder is not found, the setup cell stops with a message saying where it looked.

## 0. Setup

Imports, paths, the random seed and the **parameters of the cleaning**. Every threshold lives in this one cell, so a change made here propagates through the whole notebook.

The notebook finds the raw data in this order: the environment variable `YELP_DATA_RAW`, then `data/raw/`, then `Yelp JSON/yelp_dataset/` (where the files currently sit on our machines).

In [ ]:
import difflib
import json
import os
import platform
import random
import time
from collections import Counter
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
import networkx as nx
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

# ---------------- Paths (relative to the project root) ----------------
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

def find_data_raw() -> Path:
    # First folder that contains the Yelp business file.
    candidates = [os.environ.get("YELP_DATA_RAW"),
                  PROJECT_ROOT / "data" / "raw",
                  PROJECT_ROOT / "Yelp JSON" / "yelp_dataset"]
    for c in candidates:
        if c and (Path(c) / "yelp_academic_dataset_business.json").exists():
            return Path(c).resolve()
    looked = "\n  ".join(str(c) for c in candidates if c)
    raise FileNotFoundError("Yelp JSON files not found. Put the 3 files in data/raw/ or set YELP_DATA_RAW "
                            f"(see 'Before you run' at the top). Looked in:\n  {looked}")

DATA_RAW = find_data_raw()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "figures"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
raw_file = lambda name: DATA_RAW / f"yelp_academic_dataset_{name}.json"

# ---------------- Cleaning parameters (change them here only) ----------------
BUSINESS_CATEGORY = "Restaurants"  # keep only businesses with this Yelp category (None = all businesses)
MIN_REVIEWS = 10                   # keep users with at least this many reviewed restaurants in the chosen city
POSITIVE_MIN_STARS = 3.5           # a review is "positive" if stars >= 3.5 (review stars are whole numbers -> 4 or 5)
N_CANDIDATE_CITIES = 8             # how many cities we compare before choosing one
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---------------- Display & plotting style ----------------
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
})

print(f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"pyarrow {pa.__version__} | networkx {nx.__version__} | matplotlib {matplotlib.__version__}")
print(f"PROJECT_ROOT   = {PROJECT_ROOT}")
print(f"DATA_RAW       = {DATA_RAW}")
print(f"DATA_PROCESSED = {DATA_PROCESSED}")

## 1a. Businesses: load, check and normalise city names

The business file is small (≈ 119 MB, 150k rows), so we load it whole.

**Why normalise city names?** The dataset has no "metro area" field, only a free-text `city` typed in by the business owners. The same city can appear as `"Philadelphia"`, `"philadelphia"` or `"Philadelphia "`. We collapse extra whitespace and capitalisation, and identify a city by **(city, state)**, because some city names exist in more than one state.

We also turn `categories` (one comma-separated string) into a Python list. Steps 3 and 4 need it as a list.

**Why restaurants only?** Yelp covers very different kinds of business: restaurants, but also dentists, car repair, hair salons, plumbers and so on. A shared dentist says little about shared *taste*, and users who review many services would look similar for reasons that have nothing to do with taste. Restricting to businesses tagged **`Restaurants`** (set by `BUSINESS_CATEGORY` in the setup cell) makes "taste" concrete: *which restaurants and cuisines a user likes*. The cuisine categories (`Italian`, `Thai`, `Vegan`, …) then become natural items for steps 3 and 4.

In [ ]:
t0 = time.perf_counter()
biz = pd.read_json(raw_file("business"), lines=True)
print(f"Loaded {len(biz):,} businesses x {biz.shape[1]} columns in {time.perf_counter() - t0:.1f} s")

# --- basic quality checks ---
print("Duplicate business_id:", biz.business_id.duplicated().sum())
print("\nMissing values per column (only columns with any):")
na = biz.isna().sum()
print(na[na > 0].to_string())

In [ ]:
def normalize_city(s):
    # 'philadelphia ' -> 'Philadelphia' (collapse whitespace, title case).
    if not isinstance(s, str):
        return None
    return " ".join(s.split()).title()

biz["city_norm"] = biz["city"].map(normalize_city)
biz["state"] = biz["state"].str.strip().str.upper()
biz["city_key"] = biz["city_norm"] + ", " + biz["state"]

# categories: "Pizza, Restaurants" -> ["Pizza", "Restaurants"]; missing -> []
biz["categories_list"] = biz["categories"].map(
    lambda s: [c.strip() for c in s.split(",") if c.strip()] if isinstance(s, str) else [])

print(f"Distinct raw city strings      : {biz.city.nunique():,}")
print(f"Distinct normalised (city,state): {biz.city_key.nunique():,}")
print(f"Businesses without categories  : {(biz.categories_list.str.len() == 0).sum():,}")
print(f"Distinct categories            : {len({c for cs in biz.categories_list for c in cs}):,}")

# Scope: restaurants only (every business whose category list contains BUSINESS_CATEGORY)
biz["in_scope"] = biz["categories_list"].map(lambda cs: BUSINESS_CATEGORY is None or BUSINESS_CATEGORY in cs)
scope_ids = set(biz.loc[biz.in_scope, "business_id"])
print(f"Businesses tagged '{BUSINESS_CATEGORY}'   : {len(scope_ids):,} ({len(scope_ids) / len(biz):.1%})")

(biz.groupby("city_key").agg(businesses=("business_id", "size"), restaurants=("in_scope", "sum"))
    .sort_values("restaurants", ascending=False).head(15))

> **What we found.**
> - 150,346 businesses with no duplicate IDs. Normalisation merged the 1,416 raw city strings into 1,283 (city, state) pairs.
> - **52,268 businesses (35%) are restaurants.** **Philadelphia, PA** has the most of them by far: 5,855 restaurants, about twice as many as Tampa (2,967).
> - Only 103 businesses have no categories, and they are excluded anyway because they cannot be identified as restaurants.
> - `attributes` and `hours` have missing values, but this project does not use them.

## 1b. Reviews: one streaming pass, keep only what we need

The review file is **5.3 GB with ≈ 7M reviews**, and most of its bytes are the review `text`, which this project does not use. We read it **line by line** (one JSON object per line). From each review we keep only `review_id, user_id, business_id, stars, date`, and write the result to Parquet in chunks of 500k rows, so memory use stays small.

The result is cached as `data/processed/reviews_slim_all.parquet` (a few hundred MB). If that file already exists, the cell skips the slow pass. Delete the file to force a re-run.

In [ ]:
REVIEWS_SLIM = DATA_PROCESSED / "reviews_slim_all.parquet"
REVIEW_SCHEMA = pa.schema([("review_id", pa.string()), ("user_id", pa.string()),
                           ("business_id", pa.string()), ("stars", pa.float32()),
                           ("date", pa.timestamp("s"))])

def stream_reviews_to_parquet(src: Path, dst: Path, chunk: int = 500_000) -> Counter:
    # Stream the review JSON-lines file and write the slim columns to Parquet.
    stats = Counter()
    cols = {k: [] for k in ("review_id", "user_id", "business_id", "stars", "date")}
    tmp = dst.with_suffix(".tmp")
    writer = pq.ParquetWriter(tmp, REVIEW_SCHEMA, compression="zstd")

    def flush():
        if not cols["review_id"]:
            return
        dates = pc.strptime(pa.array(cols["date"], pa.string()), format="%Y-%m-%d %H:%M:%S", unit="s")
        table = pa.table({"review_id": cols["review_id"], "user_id": cols["user_id"],
                          "business_id": cols["business_id"],
                          "stars": pa.array(cols["stars"], pa.float32()), "date": dates},
                         schema=REVIEW_SCHEMA)
        writer.write_table(table)
        for v in cols.values():
            v.clear()

    with open(src, "rb") as f:
        for line in f:
            stats["lines"] += 1
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                stats["bad_json"] += 1
                continue
            for k in cols:
                v = r.get(k)
                if v is None:
                    stats[f"missing_{k}"] += 1
                cols[k].append(v)
            if len(cols["review_id"]) >= chunk:
                flush()
    flush()
    writer.close()
    tmp.rename(dst)  # only appears under its final name once complete
    return stats

if REVIEWS_SLIM.exists():
    print(f"Using cached {REVIEWS_SLIM.name} ({REVIEWS_SLIM.stat().st_size / 1e6:.0f} MB)")
else:
    t0 = time.perf_counter()
    stream_stats = stream_reviews_to_parquet(raw_file("review"), REVIEWS_SLIM)
    print(f"Streamed {stream_stats['lines']:,} lines in {time.perf_counter() - t0:.0f} s -> "
          f"{REVIEWS_SLIM.stat().st_size / 1e6:.0f} MB")
    print("Problems found while parsing:", {k: v for k, v in stream_stats.items() if k != "lines"} or "none")

Next we load the slim table and run **quality checks on all reviews**: unique review IDs, missing values, valid star values, date range, and reviews that point to a business missing from the business file.

To save memory, user and business IDs are loaded as pandas `category` dtype. Each distinct ID is then stored once and the rows hold small integer codes.

In [ ]:
t0 = time.perf_counter()
tbl = pq.read_table(REVIEWS_SLIM)
n_reviews_raw = tbl.num_rows
print(f"Reviews                 : {n_reviews_raw:,}")
print(f"Distinct review_id      : {pc.count_distinct(tbl['review_id']).as_py():,}")
print("Null values per column  :", {c: tbl[c].null_count for c in tbl.column_names})

reviews_all = tbl.drop_columns(["review_id"]).to_pandas(strings_to_categorical=True)
del tbl
print(f"Loaded into pandas in {time.perf_counter() - t0:.1f} s, "
      f"{reviews_all.memory_usage(deep=True).sum() / 1e6:.0f} MB in memory")

print("\nStar values:", reviews_all.stars.value_counts().sort_index().to_dict())
print("Date range :", reviews_all.date.min(), "->", reviews_all.date.max())
unknown_biz = ~reviews_all.business_id.isin(biz.business_id)
print(f"Reviews of businesses not in business file: {unknown_biz.sum():,}")
print(f"Distinct users with >= 1 review: {reviews_all.user_id.nunique():,}")

> **What we found.** The review file is clean:
> - 6,990,280 reviews; every `review_id` is unique.
> - No missing values and no malformed lines.
> - Stars are whole numbers from 1 to 5, and dates run from Feb 2005 to Jan 2022.
> - Every review points to a known business.
>
> Ratings are strongly skewed: **46% of all reviews give 5 stars**. This is one reason to normalise per user (section 1e).
>
> One quirk: the reviews contain 1,987,929 distinct users, while the user file has 1,987,897 records, so **32 reviewers have no profile**. Section 1g checks that none of our final users is affected.

## 1c. Choosing the city

Filtering to **one city** keeps the problem a manageable size. It also makes the friend graph meaningful: friends in the same city can visit the same businesses, so shared taste is actually observable.

A city is only useful to us if, **after** our filters, it still has a large, connected friend graph of active reviewers. All counts below therefore use **restaurant reviews only**. For each city we compute:
1. restaurant reviews, restaurants and users (from the review file);
2. **users with ≥ `MIN_REVIEWS` distinct reviewed restaurants _in that city_**. We count reviews inside the dataset and the city, not Yelp's `review_count` field. That field includes reviews that are not in this dataset, are in other cities or are of other kinds of business, which tell us nothing about restaurant taste here;
3. for the top `N_CANDIDATE_CITIES` cities by (2), the **friend graph among those users** and the size of its **largest connected component (LCC)**. This is the number of users we would actually analyse.

In [ ]:
t0 = time.perf_counter()
# City of every in-scope business; reviews of other businesses get no city and drop out of the counts.
city_of_biz = biz.loc[biz.in_scope].set_index("business_id")["city_key"]
reviews_all["city_key"] = reviews_all["business_id"].map(city_of_biz).astype("category")
print(f"Restaurant reviews in the whole dataset: {reviews_all.city_key.notna().sum():,} "
      f"({reviews_all.city_key.notna().mean():.1%} of all reviews)")

# distinct restaurants reviewed per (city, user)
user_city = (reviews_all[["city_key", "user_id", "business_id"]].drop_duplicates()
             .groupby(["city_key", "user_id"], observed=True).size().rename("n_biz").reset_index())

g = reviews_all.groupby("city_key", observed=True)
city_stats = pd.DataFrame({"reviews": g.size(),
                           "restaurants": g["business_id"].nunique(),
                           "users": g["user_id"].nunique()})
city_stats[f"users_ge{MIN_REVIEWS}"] = (user_city[user_city.n_biz >= MIN_REVIEWS]
                                        .groupby("city_key", observed=True).size())
city_stats = city_stats.fillna(0).astype(int).sort_values(f"users_ge{MIN_REVIEWS}", ascending=False)
print(f"Computed in {time.perf_counter() - t0:.1f} s")
city_stats.head(15)

> **What we found.** Restaurants account for **68% of all reviews** (4.72M). **Philadelphia** has the most restaurant reviews (688k) and by far the most active restaurant reviewers: 10,731 users with ≥ 10 restaurant reviews, more than twice the runner-up New Orleans (4,836). The activity filter is harsh everywhere. Only about 5% of a city's restaurant reviewers have ≥ 10 reviews there, because most reviewers are occasional users or tourists.

To measure the friend graphs we need the `friends` field of the user file (3.4 GB, ≈ 2M users). We stream it once. We keep the full record (friend list plus a few profile fields) **only for users who pass the review threshold in one of the candidate cities**. For everyone else we only update global counters.

The friend list is a comma-separated string of user IDs, or the literal string `"None"`. The kept records are cached as `data/processed/users_candidates.parquet`.

In [ ]:
candidate_cities = list(city_stats.index[:N_CANDIDATE_CITIES])
active = user_city[(user_city.n_biz >= MIN_REVIEWS) & user_city.city_key.isin(candidate_cities)]
candidate_users = {c: set(active.loc[active.city_key == c, "user_id"].astype(str)) for c in candidate_cities}
all_candidates = set().union(*candidate_users.values())
print("Candidate cities:", candidate_cities)
print(f"Active users across candidate cities: {len(all_candidates):,}")

USERS_CAND = DATA_PROCESSED / "users_candidates.parquet"
KEEP_USER_FIELDS = ["user_id", "review_count", "yelping_since", "fans", "average_stars"]

def parse_friends(s):
    if not s or s == "None":
        return []
    return [x.strip() for x in s.split(",") if x.strip()]

def stream_users(src: Path, keep_ids: set):
    # Stream the user file; return kept records and global counters.
    rows, stats = [], Counter()
    with open(src, "rb") as f:
        for line in f:
            r = json.loads(line)
            stats["users"] += 1
            friends = parse_friends(r.get("friends"))
            stats["users_without_friends"] += (len(friends) == 0)
            stats["friend_links_listed"] += len(friends)
            if r["user_id"] in keep_ids:
                rec = {k: r.get(k) for k in KEEP_USER_FIELDS}
                rec["friends"] = friends
                rows.append(rec)
    return pd.DataFrame(rows), stats

USER_STATS = DATA_PROCESSED / "users_global_stats.json"
if (USERS_CAND.exists() and USER_STATS.exists()
        and set(pd.read_parquet(USERS_CAND, columns=["user_id"]).user_id) >= all_candidates):
    users_cand = pd.read_parquet(USERS_CAND)
    user_stats = json.loads(USER_STATS.read_text())
    print(f"Using cached {USERS_CAND.name}: {len(users_cand):,} users")
else:
    t0 = time.perf_counter()
    users_cand, user_stats = stream_users(raw_file("user"), all_candidates)
    users_cand.to_parquet(USERS_CAND, index=False)
    USER_STATS.write_text(json.dumps(dict(user_stats)))
    print(f"Streamed {user_stats['users']:,} users in {time.perf_counter() - t0:.0f} s; kept {len(users_cand):,}")
print(f"Users in the user file: {user_stats['users']:,}; with no friends at all: "
      f"{user_stats['users_without_friends']:,} ({user_stats['users_without_friends'] / user_stats['users']:.1%})")

missing = all_candidates - set(users_cand.user_id)
print(f"Active users missing from the user file: {len(missing):,}")
friends_of = dict(zip(users_cand.user_id, users_cand.friends))

> **What we found.** 44% of all Yelp users list no friends at all. Every active user in the candidate cities has a record in the user file.

For each candidate city we build the **undirected friendship graph among its active users**. An edge links two users who both pass the threshold in that city and list each other (or one lists the other) as friends. We also check whether friendships are **symmetric** in the data, i.e. whether *u* listing *v* implies *v* lists *u*.

In [ ]:
def build_friend_graph(user_ids, friends_of):
    # Undirected friendship graph restricted to `user_ids`, plus the share of reciprocated links.
    S = set(user_ids)
    directed = {(u, v) for u in S for v in friends_of.get(u, ()) if v in S and v != u}
    reciprocated = sum((v, u) in directed for (u, v) in directed) / max(len(directed), 1)
    G = nx.Graph()
    G.add_nodes_from(S)
    G.add_edges_from(directed)  # (u,v) and (v,u) collapse into one undirected edge
    return G, reciprocated

t0 = time.perf_counter()
rows = []
for c in candidate_cities:
    G, recip = build_friend_graph(candidate_users[c], friends_of)
    lcc = max(nx.connected_components(G), key=len)
    rows.append({"city": c, "active_users": G.number_of_nodes(), "friend_edges": G.number_of_edges(),
                 "isolated_users": nx.number_of_isolates(G), "reciprocated_links": recip,
                 "lcc_users": len(lcc), "lcc_share": len(lcc) / G.number_of_nodes(),
                 "lcc_edges": G.subgraph(lcc).number_of_edges(),
                 "lcc_avg_degree": 2 * G.subgraph(lcc).number_of_edges() / len(lcc)})
candidates = pd.DataFrame(rows).set_index("city").sort_values("lcc_users", ascending=False)
print(f"Built {len(candidate_cities)} graphs in {time.perf_counter() - t0:.1f} s")
candidates.style.format({"reciprocated_links": "{:.1%}", "lcc_share": "{:.1%}", "lcc_avg_degree": "{:.1f}",
                         "active_users": "{:,}", "friend_edges": "{:,}", "isolated_users": "{:,}",
                         "lcc_users": "{:,}", "lcc_edges": "{:,}"})

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
order = candidates.sort_values("active_users")
y = np.arange(len(order))
ax.barh(y + 0.2, order.active_users, height=0.38, color=BLUE, label=f"Users with ≥ {MIN_REVIEWS} restaurant reviews")
ax.barh(y - 0.2, order.lcc_users, height=0.38, color=ORANGE, label="…of which in the friend-graph LCC")
ax.set_yticks(y, order.index)
ax.set_xlabel("Users")
ax.xaxis.set_major_formatter(matplotlib.ticker.StrMethodFormatter("{x:,.0f}"))
ax.set_title("Active users and largest friend component per candidate city")
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="y", visible=False)
fig.savefig(FIGURES / "01_city_candidates.png")
plt.show()

> **What we found.**
> - **Friendships are symmetric**: 100% of links are reciprocated in every city, so we can safely treat the friend graph as undirected.
> - In every city, 60–86% of the active users fall into one large connected component. The rest are almost all **isolated users**, people with no friend among the city's active restaurant reviewers.
> - Philadelphia's LCC (≈ 7.9k users, ≈ 68k friendships, mean degree 17) is **more than twice as large as any other city's**. Indianapolis and New Orleans come next with about 3.4k each.

**Decision.** We pick the city whose **largest connected component is largest**: it gives the most users for whom both questions, friend community and taste community, can be answered. The cell below makes the choice from the table above. To try another city, override `CITY` there and re-run everything below it.

We also check for **misspelled variants** of the chosen city name in the same state that the simple normalisation did not merge, and merge the ones that clearly refer to the same city.

In [ ]:
CITY = candidates["lcc_users"].idxmax()   # override here, e.g. CITY = "Tampa, FL"
city_name, city_state = CITY.rsplit(", ", 1)
CITY_SLUG = f"{city_name}_{city_state}".lower().replace(" ", "_")
print(f"Chosen city: {CITY}   (file prefix: {CITY_SLUG})\n")

# spelling variants in the same state that normalisation did not merge
same_state = biz[biz.state == city_state]
variants = difflib.get_close_matches(city_name, same_state.city_norm.dropna().unique(), n=15, cutoff=0.75)
var_counts = same_state[same_state.city_norm.isin(variants)].groupby("city_norm").size().sort_values(ascending=False)
print("Close spellings in the same state (businesses):")
print(var_counts.to_string())

# All of these are the same city (typos, "Pa" suffix, a neighbourhood name), so we merge them.
# If you override CITY, inspect this list and remove any name that is really a different place.
CITY_NAMES = set(variants)
city_biz_df = biz[(biz.state == city_state) & biz.city_norm.isin(CITY_NAMES)]
city_biz = set(city_biz_df.business_id)
print(f"\nBusinesses in {CITY} after merging variants: {len(city_biz):,} "
      f"(+{len(city_biz) - (biz.city_key == CITY).sum()} from variant spellings)")

> **What we found.** We choose **Philadelphia, PA**. Its spelling variants (`Philadephia`, `Philadelphia Pa`, `Philiadelphia`, …, and the neighbourhood name `West Philadelphia`) add only 13 businesses, of which a single restaurant. We merge them anyway for correctness, although the effect is negligible.

## 1d. Filtering the chosen city

We apply the filters **in this order** and log reviews / users / businesses after each one:

1. **City**: reviews of businesses in the chosen city (including the merged spelling variants).
2. **Restaurants**: keep only reviews of businesses tagged `Restaurants` (see 1a).
3. **One review per (user, restaurant)**: a user can review the same restaurant more than once. For "the set of restaurants a user reviewed" a repeat adds nothing, and it would let a user reach the threshold by reviewing one place ten times. We keep the **most recent** review, which is the user's latest opinion.
4. **Users with ≥ `MIN_REVIEWS` restaurant reviews**: users with very few reviews have tiny taste sets. Their Jaccard similarities are noisy, and their mean rating (used for normalisation) is unreliable.
5. **Largest connected component of the friendship graph** among the remaining users. Communities can only be detected among connected users. Isolated users and tiny islands carry no friend-community information.

This order matters:
- The restaurant filter comes **before** the activity filter, so the 10 reviews must all be restaurant reviews.
- The friend graph is built **after** the activity filter, so every node in the final graph is an active restaurant reviewer.
- Step 5 only removes users and never removes reviews of kept users, so everyone in the LCC still has ≥ `MIN_REVIEWS` reviews. No second filtering pass is needed.

In [ ]:
funnel = []
def log_step(step, reviews, n_edges=None):
    funnel.append({"step": step, "reviews": len(reviews), "users": reviews.user_id.nunique(),
                   "businesses": reviews.business_id.nunique(), "friend_edges": n_edges})

log_step("0. all reviews (whole dataset)", reviews_all)

# 1. city -------------------------------------------------------------
rv = reviews_all[reviews_all.business_id.isin(list(city_biz))].drop(columns="city_key").copy()
rv["user_id"] = rv["user_id"].astype(str)
rv["business_id"] = rv["business_id"].astype(str)
log_step(f"1. reviews in {CITY}", rv)

# 2. restaurants only ----------------------------------------------------
rv = rv[rv.business_id.isin(scope_ids)].reset_index(drop=True)
log_step(f"2. only '{BUSINESS_CATEGORY}'", rv)

# 3. one review per (user, business): keep the most recent --------------
dup_pairs = rv.duplicated(["user_id", "business_id"], keep=False)
print(f"Reviews that are repeats of the same (user, restaurant): {dup_pairs.sum():,} "
      f"in {rv[dup_pairs].groupby(['user_id', 'business_id']).ngroups:,} pairs")
rv = (rv.sort_values(["user_id", "business_id", "date"])
        .drop_duplicates(["user_id", "business_id"], keep="last")
        .reset_index(drop=True))
log_step("3. one review per (user, restaurant)", rv)

# 4. users with >= MIN_REVIEWS reviews -----------------------------------
n_per_user = rv.groupby("user_id").size()
active_users = set(n_per_user[n_per_user >= MIN_REVIEWS].index)
rv = rv[rv.user_id.isin(active_users)].reset_index(drop=True)
log_step(f"4. users with >= {MIN_REVIEWS} restaurant reviews", rv)

# Active users whose friend list we have not read yet (e.g. pushed over the threshold by the
# merged spelling variants, or a CITY override): stream the user file once more for them.
missing_profiles = active_users - set(users_cand.user_id)
if missing_profiles:
    extra, _ = stream_users(raw_file("user"), missing_profiles)
    users_cand = pd.concat([users_cand, extra], ignore_index=True)
    users_cand.to_parquet(USERS_CAND, index=False)
    friends_of.update(zip(extra.user_id, extra.friends))
print(f"Active users whose profile had to be fetched additionally: {len(missing_profiles):,}")

# 5. friendship graph + largest connected component ----------------------
G_all, recip = build_friend_graph(active_users, friends_of)
components = sorted(nx.connected_components(G_all), key=len, reverse=True)
lcc_nodes = components[0]
G = G_all.subgraph(lcc_nodes).copy()
rv = rv[rv.user_id.isin(lcc_nodes)].reset_index(drop=True)
log_step("5. largest connected component", rv, n_edges=G.number_of_edges())

funnel_df = pd.DataFrame(funnel).set_index("step")
funnel_df["users_kept_vs_prev"] = funnel_df.users / funnel_df.users.shift()
funnel_df.style.format({"reviews": "{:,}", "users": "{:,}", "businesses": "{:,}",
                        "friend_edges": lambda v: "" if pd.isna(v) else f"{int(v):,}",
                        "users_kept_vs_prev": lambda v: "" if pd.isna(v) else f"{v:.1%}"})

> **What we found.** Reading the funnel row by row:
> 1. **City**: Philadelphia holds 968,388 reviews, 14% of the dataset.
> 2. **Restaurants**: 687,827 of those reviews (71%) are of 5,856 restaurants. 75% of the city's reviewers have reviewed at least one restaurant.
> 3. **Repeats**: 40,876 reviews are repeats of the same (user, restaurant), in 19,331 pairs. Keeping one review per pair removes 21,545 reviews.
> 4. **Activity**: the ≥ 10 restaurant reviews filter is the strongest. It keeps only **5.1% of users** (10,732 of 209,694), but **44% of the reviews**, because the activity is concentrated in a small core of heavy users.
> 5. **LCC**: keeps 74% of the active users and 85% of their reviews.
>
> **Final dataset: 7,947 users, 67,622 friendships, 251,375 reviews and 5,759 restaurants.** Every user is an active restaurant reviewer connected to all the others through friendships.

What the friend graph looked like **before** taking the LCC, i.e. which users step 5 removes:

In [ ]:
sizes = pd.Series([len(c) for c in components])
print(f"Active users                 : {G_all.number_of_nodes():,}")
print(f"Friend edges among them      : {G_all.number_of_edges():,}")
print(f"Reciprocated friend links    : {recip:.1%}  (share of u->v links where v->u is also listed)")
print(f"Connected components         : {len(components):,}")
print(f"  of which isolated users    : {(sizes == 1).sum():,}")
print(f"  largest component          : {sizes.iloc[0]:,} users ({sizes.iloc[0] / sizes.sum():.1%})")
print(f"  second largest component   : {sizes.iloc[1] if len(sizes) > 1 else 0:,} users")
print("Component size distribution (size: count):", sizes.value_counts().sort_index().head(10).to_dict())

> **What we found.** The users removed by the LCC step are almost all **isolated users** (2,725), people with no friend among Philadelphia's active restaurant reviewers. The rest are only 27 pairs and 2 triples. There is no second large component, so restricting to the LCC loses no community structure.

**Possible bias:** the kept users are the most active and socially connected part of Yelp. Our conclusions apply to this group, not to Yelp users in general. We should mention this in the limitations.

## 1e. Normalising star ratings per user

People use the 1–5 star scale differently. One user gives 5 stars to everything they like, another rarely gives more than 4. To compare **taste** rather than **rating habits**, we centre each user's ratings on their own mean:

$$\tilde r_{u,b} = r_{u,b} - \bar r_u ,\qquad \bar r_u = \frac{1}{|B_u|}\sum_{b\in B_u} r_{u,b}$$

where $B_u$ is the set of businesses user $u$ reviewed **in the cleaned data**. A positive $\tilde r$ means "this user liked this business more than they usually like places". The mean is computed on the cleaned reviews (the chosen city, one review per business), so it matches exactly the data used later.

We also keep each user's standard deviation. Users who give the **same rating to everything** (std = 0) end up with all-zero normalised ratings. They still have a valid set of reviewed businesses (useful for MinHash/LSH), but their ratings contain no preference information. We flag them rather than drop them.

In [ ]:
stats_u = rv.groupby("user_id")["stars"].agg(n_reviews="size", mean_stars="mean", std_stars="std")
stats_u["std_stars"] = stats_u["std_stars"].fillna(0.0)
rv = rv.merge(stats_u[["mean_stars"]], left_on="user_id", right_index=True)
rv["stars_norm"] = (rv["stars"] - rv["mean_stars"]).astype("float32")
rv = rv.drop(columns="mean_stars")

flat = (stats_u.std_stars == 0).sum()
print(f"Users whose ratings are all identical (std = 0): {flat:,} ({flat / len(stats_u):.1%})")
print(f"Mean of per-user means: {stats_u.mean_stars.mean():.2f} stars "
      f"(range {stats_u.mean_stars.min():.2f} – {stats_u.mean_stars.max():.2f})")
print(f"Max |mean of normalised ratings| over users: "
      f"{rv.groupby('user_id').stars_norm.mean().abs().max():.2e}  (should be ~0)")

### Positive ratings (for the baskets in step 4)

Step 4 treats each user as a **basket of the restaurants (or cuisines) they rated positively**. We fix the definition here, so every later step uses the same one:

$$\text{positive}(u,b) \iff r_{u,b} \ge 3.5 \quad(\texttt{POSITIVE\_MIN\_STARS})$$

Review stars are whole numbers, so in practice this means **4 or 5 stars**. This is an *absolute* threshold: "the user liked it". We store it as a boolean column `is_positive` next to the raw and normalised stars. The normalised ratings stay available for the parts that compare rating *profiles* (e.g. the optional k-means on per-category rating vectors).

We also check what the threshold does to the baskets. A user with no positive review has an empty basket, and a user with a single positive review cannot contribute to any frequent *pair*.

In [ ]:
rv["is_positive"] = rv["stars"] >= POSITIVE_MIN_STARS
pos_per_user = rv.groupby("user_id")["is_positive"].sum()

print(f"Positive reviews (stars >= {POSITIVE_MIN_STARS}): {rv.is_positive.sum():,} of {len(rv):,} ({rv.is_positive.mean():.1%})")
print(f"Positive restaurants per user: median {pos_per_user.median():.0f}, mean {pos_per_user.mean():.1f}")
print(f"Users with 0 positive reviews : {(pos_per_user == 0).sum():,}")
print(f"Users with < 2 positive reviews: {(pos_per_user < 2).sum():,} ({(pos_per_user < 2).mean():.1%})")
agree = (rv.is_positive == (rv.stars_norm > 0)).mean()
print(f"Agreement with the relative definition (stars_norm > 0): {agree:.1%} of reviews")

> **What we found.**
> - **69.5% of the reviews are positive** (4 or 5 stars). The median user has 14 positive restaurants, the mean is 22.
> - Almost every user has a usable basket: only 6 users have no positive review and 16 (0.2%) have fewer than 2. We keep them, because they still matter for the friend graph and the taste sets, and step 4 simply skips empty or single-item baskets.
> - The absolute threshold agrees with the relative one (`stars_norm > 0`) on 86.5% of reviews. Almost all disagreements (32k of 34k) are **4-star reviews by generous raters**: for a user who averages 4.5 stars, a 4 is below their usual rating, yet it still counts as positive here. This is a known consequence of the absolute definition, and worth one sentence in the report.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
vc = rv.stars.value_counts().sort_index()
axes[0].bar(vc.index, vc.values, width=0.6, color=BLUE)
axes[0].set(title="Raw stars", xlabel="Stars", ylabel="Reviews")
axes[1].hist(stats_u.mean_stars, bins=40, color=BLUE, edgecolor="white", linewidth=0.5)
axes[1].set(title="Per-user mean rating", xlabel="Mean stars of a user", ylabel="Users")
axes[2].hist(rv.stars_norm, bins=np.arange(-4.25, 4.3, 0.25), color=BLUE, edgecolor="white", linewidth=0.5)
axes[2].set(title="Normalised stars (stars − user mean)", xlabel="Normalised stars", ylabel="Reviews")
for ax in axes:
    ax.yaxis.set_major_formatter(matplotlib.ticker.StrMethodFormatter("{x:,.0f}"))
fig.tight_layout()
fig.savefig(FIGURES / "01_star_normalisation.png")
plt.show()

> **What we found.** Even among active users the per-user mean rating varies a lot. The average user mean is 3.88 stars, but individual means range from 1 to 5. Two users can therefore give the same 4 stars with opposite meanings, and this is exactly what the normalisation corrects.
>
> After centring, ratings spread roughly from −3.9 to +3.8 around 0, with a longer **negative tail**: a user's bad experiences stand out more strongly from their usual rating than their good ones. 57% of reviews are above the user's own mean.
>
> Only 36 users (0.5%) rate everything the same. We keep them, since their restaurant sets are still informative.

## 1f. A first look at the cleaned data

Three properties of the cleaned data matter directly for the later steps:
- **Reviews per user**: the size of each user's taste set (step 3, MinHash).
- **Friend degree**: how dense the graph is that Louvain will partition (step 2).
- **Reviewers per restaurant**: if most restaurants are reviewed by only one kept user, user–restaurant sets overlap little, so Jaccard similarities will be small. That is an argument for also trying **categories (cuisines)** instead of restaurants in step 3.

In [ ]:
def int_log_bins(max_value, start=1, n=30):
    # Log-spaced bin edges rounded to integers (avoids empty bins for small integer values).
    return np.unique(np.round(np.logspace(np.log10(start), np.log10(max_value + 1), n))) - 0.5

deg = pd.Series(dict(G.degree()), name="degree")
reviewers_per_biz = rv.groupby("business_id").size()

fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
axes[0].hist(stats_u.n_reviews, bins=int_log_bins(stats_u.n_reviews.max(), start=MIN_REVIEWS), color=BLUE, edgecolor="white", linewidth=0.5)
axes[0].set(xscale="log", yscale="log", title="Reviews per user", xlabel="Reviews (log)", ylabel="Users (log)")
axes[1].hist(deg, bins=int_log_bins(deg.max()), color=BLUE, edgecolor="white", linewidth=0.5)
axes[1].set(xscale="log", yscale="log", title="Friends per user (in LCC)", xlabel="Degree (log)", ylabel="Users (log)")
axes[2].hist(reviewers_per_biz, bins=int_log_bins(reviewers_per_biz.max()), color=BLUE, edgecolor="white", linewidth=0.5)
axes[2].set(xscale="log", yscale="log", title="Kept reviewers per restaurant", xlabel="Reviewers (log)", ylabel="Restaurants (log)")
fig.tight_layout()
fig.savefig(FIGURES / "01_cleaned_distributions.png")
plt.show()

print(f"Reviews per user : median {stats_u.n_reviews.median():.0f}, "
      f"mean {stats_u.n_reviews.mean():.1f}, max {stats_u.n_reviews.max():,}")
print(f"Friend degree    : median {deg.median():.0f}, mean {deg.mean():.1f}, max {deg.max():,}; "
      f"graph density {nx.density(G):.2e}")
print(f"Restaurants reviewed by exactly 1 kept user: {(reviewers_per_biz == 1).mean():.1%} "
      f"(they hold {reviewers_per_biz[reviewers_per_biz == 1].sum() / len(rv):.1%} of reviews)")

> **What we found.** All three distributions are **heavy-tailed**:
> - **Reviews per user**: the median user has 19 restaurant reviews, the mean is 32 and the maximum is 706.
> - **Friends per user**: the median user has 5 friends in the LCC, but a few hub users have thousands (max 2,832). The graph is sparse (density 2.1·10⁻³), which suits Louvain. Hubs can glue communities together, so step 2 should look at them.
> - **Reviewers per restaurant**: only 4.4% of restaurants are reviewed by a single kept user, holding 0.1% of the reviews. The user–restaurant sets therefore overlap well, and Jaccard/MinHash should find meaningful similar pairs.

In [ ]:
# Categories of the kept restaurants (what steps 3–4 can use instead of restaurant IDs).
# The scope tag itself ("Restaurants") is on every business now, so it carries no information:
# `categories_taste` is the category list without it.
biz_city = biz[biz.business_id.isin(rv.business_id)].copy()
biz_city["categories_taste"] = biz_city["categories_list"].map(lambda cs: [c for c in cs if c != BUSINESS_CATEGORY])

user_cats = (rv[["user_id", "business_id"]]
             .merge(biz_city[["business_id", "categories_taste"]], on="business_id")
             .explode("categories_taste").dropna(subset=["categories_taste"])
             .drop_duplicates(["user_id", "categories_taste"]))
cats_per_user = user_cats.groupby("user_id").size()
cat_table = pd.DataFrame({
    "restaurants": Counter(c for cs in biz_city.categories_taste for c in cs),
    "share_of_users": user_cats.groupby("categories_taste").size() / rv.user_id.nunique(),
}).sort_values("share_of_users", ascending=False)

print(f"Distinct categories among kept restaurants (without '{BUSINESS_CATEGORY}'): {len(cat_table):,}")
print(f"Distinct categories per user: median {cats_per_user.median():.0f}, mean {cats_per_user.mean():.0f}")
print(f"Categories reached by > 50% of users: {(cat_table.share_of_users > 0.5).sum()}, "
      f"> 25%: {(cat_table.share_of_users > 0.25).sum()}")
closed = biz_city.loc[biz_city.is_open == 0, "business_id"]
print(f"Closed restaurants (is_open = 0): {len(closed) / len(biz_city):.1%} of restaurants, "
      f"{rv.business_id.isin(closed).mean():.1%} of reviews (kept: the reviews still show taste)")
cat_table.head(25).style.format({"restaurants": "{:,}", "share_of_users": "{:.1%}"})

> **What we found.** Without the `Restaurants` tag, the kept restaurants carry **374 categories**, and the median user touches 48 of them.
>
> Even within restaurants, Yelp's category list mixes **cuisines** (`Italian`, `Chinese`, `Mexican`, `Vegetarian`, …) with **generic tags** (`Food` 99% of users, `Nightlife` 98%, `Bars` 97%, `Event Planning & Services` 67%). **36 categories reach more than half of all users.** Category sets are therefore dense, and Jaccard similarity on them will be high for nearly everyone.
>
> For category-based taste (steps 3–4) we should **drop the most widespread categories**, for example those reaching more than 50% of users. Restaurant-level sets (`business_id`) are the cleaner primary representation.
>
> Closed restaurants are 40% of restaurants and 29% of reviews. We keep them: a review of a now-closed restaurant still shows what the user liked.

## 1g. Sanity checks and saving

Before saving, we **assert** the properties that later steps rely on. If any of them fails, the notebook stops here instead of silently passing bad data on.

In [ ]:
kept_users = set(rv.user_id)
assert kept_users == set(G.nodes), "users in reviews and in the graph must be identical"
assert nx.is_connected(G), "the friendship graph must be one connected component"
assert rv.groupby("user_id").size().min() >= MIN_REVIEWS, "every user needs >= MIN_REVIEWS reviews"
assert not rv.duplicated(["user_id", "business_id"]).any(), "one review per (user, business)"
assert rv.business_id.isin(city_biz).all(), "all businesses are in the chosen city"
assert rv.business_id.isin(scope_ids).all(), "all businesses are restaurants"
assert (rv.is_positive == (rv.stars >= POSITIVE_MIN_STARS)).all(), "is_positive follows the threshold"
assert rv.stars.between(1, 5).all(), "stars are in 1..5"
assert np.allclose(rv.groupby("user_id").stars_norm.mean(), 0, atol=1e-5), "normalised ratings average 0 per user"
assert kept_users <= set(users_cand.user_id), "every kept user has a profile in the user file"
print("All checks passed.")

We save four tables, prefixed with the city (`<city>` = the `CITY_SLUG` printed above):

| File | One row per | Columns |
|---|---|---|
| `<city>_users.parquet` | user in the LCC | `user_id`, `n_reviews`, `n_positive`, `mean_stars`, `std_stars` (from the cleaned reviews), `degree` (friends in the LCC), and profile fields from the user file: `yelp_review_count`, `yelping_since`, `fans`, `yelp_average_stars` |
| `<city>_friend_edges.parquet` | undirected friendship | `u`, `v` with `u < v` |
| `<city>_reviews.parquet` | (user, restaurant) | `user_id`, `business_id`, `stars`, `stars_norm`, `is_positive` (stars ≥ 3.5), `date` |
| `<city>_businesses.parquet` | restaurant with ≥ 1 kept review | `business_id`, `name`, `city`, `state`, `latitude`, `longitude`, `stars`, `review_count`, `is_open`, `categories` (full list), `categories_taste` (without `Restaurants`), `n_reviews_clean` |

`cleaning_summary.json` records the parameters and the counts after every filter, so the report can quote them.

In [ ]:
users_out = (stats_u.loc[sorted(kept_users)]
             .join(pos_per_user.rename("n_positive"))
             .join(deg)
             .join(users_cand.set_index("user_id")[["review_count", "yelping_since", "fans", "average_stars"]]
                   .rename(columns={"review_count": "yelp_review_count", "average_stars": "yelp_average_stars"}))
             .reset_index().rename(columns={"index": "user_id"}))
users_out["yelping_since"] = pd.to_datetime(users_out["yelping_since"])

edges_out = pd.DataFrame([tuple(sorted(e)) for e in G.edges()], columns=["u", "v"]).sort_values(["u", "v"])

reviews_out = rv[["user_id", "business_id", "stars", "stars_norm", "is_positive", "date"]].sort_values(["user_id", "date"])

biz_out = (biz_city[["business_id", "name", "city_norm", "state", "latitude", "longitude",
                     "stars", "review_count", "is_open", "categories_list", "categories_taste"]]
           .rename(columns={"categories_list": "categories"})
           .assign(city=city_name)            # canonical name (merges the spelling variants)
           .drop(columns="city_norm")
           .merge(reviewers_per_biz.rename("n_reviews_clean"), left_on="business_id", right_index=True))

outputs = {"users": users_out, "friend_edges": edges_out, "reviews": reviews_out, "businesses": biz_out}
for name, df in outputs.items():
    path = DATA_PROCESSED / f"{CITY_SLUG}_{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"{path.relative_to(PROJECT_ROOT)}: {len(df):,} rows x {df.shape[1]} cols, {path.stat().st_size / 1e6:.1f} MB")

summary = {
    "city": CITY, "business_category": BUSINESS_CATEGORY, "min_reviews": MIN_REVIEWS,
    "positive_min_stars": POSITIVE_MIN_STARS, "seed": SEED,
    "dedup_rule": "keep most recent review per (user, business)",
    "normalisation": "stars_norm = stars - mean stars of the user over the cleaned reviews",
    "positive": f"is_positive = stars >= {POSITIVE_MIN_STARS}",
    "funnel": (funnel_df.drop(columns="users_kept_vs_prev").reset_index()
               .astype(object).where(lambda d: d.notna(), None).to_dict(orient="records")),
    "final": {"users": len(users_out), "friend_edges": len(edges_out),
              "reviews": len(reviews_out), "businesses": len(biz_out)},
}
(DATA_PROCESSED / "cleaning_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("data/processed/cleaning_summary.json written")

### How later steps load the clean data
```python
import pandas as pd, networkx as nx
P = "data/processed/<city>_"            # e.g. the CITY_SLUG printed above
users   = pd.read_parquet(P + "users.parquet")
edges   = pd.read_parquet(P + "friend_edges.parquet")
reviews = pd.read_parquet(P + "reviews.parquet")
biz     = pd.read_parquet(P + "businesses.parquet")
G = nx.from_pandas_edgelist(edges, "u", "v")                        # step 2: Louvain
user_sets = reviews.groupby("user_id")["business_id"].agg(set)      # step 3: MinHash/LSH
baskets = reviews[reviews.is_positive].groupby("user_id")["business_id"].agg(list)   # step 4 (restaurants)
cuisine_of = biz.set_index("business_id")["categories_taste"]                     # step 4 (cuisines)
```

## Summary of step 1

**Final cleaned dataset (Philadelphia, PA):**

| | Count |
|---|---|
| Users (all with ≥ 10 restaurant reviews, all in one friend component) | 7,947 |
| Friendships (undirected) | 67,622 |
| Reviews (one per user–restaurant, with normalised stars and `is_positive`) | 251,375 |
| Restaurants | 5,759 |

### Decision log

| # | Decision | Reason / evidence |
|---|---|---|
| 1 | **Restaurants only** (businesses tagged `Restaurants`), applied before the activity filter | Makes "taste" concrete (food preferences) and avoids users looking similar because of shared services (dentists, car repair). Restaurants account for 68% of all reviews. *Team decision.* |
| 2 | City = **Philadelphia, PA** | Largest friend-graph LCC after filtering (≈ 7.9k users vs ≈ 3.4k for the next city); most active restaurant reviewers (10.7k). |
| 3 | A city is identified by (normalised name, state); misspelled city names are merged into Philadelphia | Free-text city field; variants listed in 1c; impact negligible (1 extra restaurant). |
| 4 | One review per (user, restaurant): keep the **most recent** | 40,876 repeat reviews in 19,331 pairs; repeats would let a user reach the threshold by reviewing one place repeatedly. |
| 5 | Users with **≥ 10 distinct restaurants reviewed in Philadelphia** (not Yelp's `review_count` field) | Taste sets must be large enough for stable Jaccard similarities and per-user means; `review_count` includes reviews outside the dataset, the city and restaurants. |
| 6 | Treat friendship as **undirected** | 100% of friend links are reciprocated. |
| 7 | Keep the **LCC of the friend graph built after the activity filter** | 74% of active users are in it; all other components have ≤ 3 users (2,725 isolated users). |
| 8 | `stars_norm = stars − user mean` over the cleaned reviews; constant raters (36) kept | Per-user means range from 1 to 5; centring separates taste from rating habit. |
| 9 | **Positive rating = stars ≥ 3.5**, i.e. 4 or 5 stars (`is_positive`) | Simple and easy to explain; 69.5% of reviews; only 16 users have fewer than 2 positives. Agrees with `stars_norm > 0` on 86.5% of reviews. *Team decision.* |
| 10 | Keep **closed restaurants** | 29% of reviews; they still show what users like. |
| 11 | Drop the review `text`, tips and check-ins | Not needed for the research question; this saves memory and time. |

### Notes for the next steps
- **Step 2 (friend communities):** load `philadelphia_pa_friend_edges.parquet`. A handful of hub users have more than 1,000 friends, so check whether they dominate any community.
- **Step 3 (taste communities):** restaurant sets overlap well (only 4% of restaurants have a single kept reviewer), so they are a good primary representation. For category sets, use `categories_taste` and drop the most widespread tags (36 reach more than 50% of users) before MinHash/LSH.
- **Step 4 (baskets):** baskets are the restaurants (or `categories_taste` of the restaurants) with `is_positive == True`. Skip the 16 users with fewer than 2 positive items.
- **Limitations to report:** the filters keep the most active and socially connected users (selection bias); the absolute positive threshold is lenient for generous raters; friendships have no timestamps; one city may not generalise.

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for f in sorted((root / "data" / "processed").glob("*.parquet")):
    pf = pq.ParquetFile(f)
    print(f"{f.name}: {pf.metadata.num_rows:,} filas")
    print("   ", pf.schema_arrow)
    print()

---
*Step 2 — from `02_friend_communities.ipynb`.*

# Step 2 — Friend communities (Louvain, implemented from scratch)

**Project question:** *Are friend communities also taste communities?*
(DTU 02807 Computational Tools for Data Science, Yelp Open Dataset)

*Code: Louvain is written in this notebook (section 2b) and explained step by step; `src/louvain.py` holds the same code (plus edge weights) for reuse in step 3.*

**Question.** Which groups of friends exist among the Philadelphia users? Step 1 built their friendship graph. Here we detect communities in it with **Louvain** and evaluate them with **modularity**. The output is one community label per user, which the later steps compare with the taste communities (NMI / ARI, permutation test).

**Plan**
- **2a** Load the cleaned friendship graph from step 1.
- **2b** **Louvain written from scratch** (local moving + aggregation), and a check of the modularity formula.
- **2c** Validate our implementation: Zachary's karate club, and our Louvain vs `networkx` on the real graph.
- **2d** Run it with 20 random seeds: modularity, number of communities, **stability** between runs.
- **2e** Effect of the **resolution parameter**, to justify the value we use.
- **2f** Analyse the final communities: sizes, density, hubs, user profile.
- **2g** **Robustness**: remove the hubs, and compare with a degree-preserving random graph.
- **2h** Save the partition to `data/processed/` for steps 3–5.

**Input:** `data/processed/philadelphia_pa_friend_edges.parquet` and `philadelphia_pa_users.parquet` (produced by `01_data_cleaning.ipynb`).
**Output:** `philadelphia_pa_friend_communities.parquet` (`user_id`, `friend_community`) and a JSON summary.

**Run:** `jupyter lab notebooks/02_friend_communities.ipynb`, then *Run → Run All Cells*. It needs `pandas numpy networkx matplotlib pyarrow scikit-learn` and takes a few minutes (the Louvain runs are pure Python).

## 0. Setup

Imports, paths, the random seed and the parameters of the analysis. Everything that can be tuned is in this cell.

In [ ]:
import json
import platform
import random
import time
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# ---------------- Paths (relative to the project root) ----------------
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
P = str(DATA_PROCESSED / "philadelphia_pa_")  # prefix of the files written by step 1

# ---------------- Parameters ----------------
N_SEEDS = 20                       # independent Louvain runs on the real graph
RESOLUTION = 1.0                   # gamma = 1 is the standard modularity used in the course
RESOLUTIONS = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]   # values compared in section 2e
SEEDS_PER_RESOLUTION = 5
HUB_QUANTILE = 0.99                # "hub" = top 1% of users by number of friends
N_NULL_GRAPHS = 3                  # degree-preserving random graphs in section 2g
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---------------- Plot style (same as step 1) ----------------
BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
})

print(f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | networkx {nx.__version__}")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

## 2a. Load the friendship graph

We load the edge list and the user table written by step 1 and build an **undirected, unweighted** graph (step 1 showed that 100% of friend links are reciprocated). The nodes are sorted so the graph, and therefore the seeded runs, are reproducible.

In [ ]:
edges = pd.read_parquet(P + "friend_edges.parquet")
users = pd.read_parquet(P + "users.parquet").set_index("user_id")

G = nx.Graph()
G.add_nodes_from(sorted(users.index))
G.add_edges_from(sorted(zip(edges.u, edges.v)))

print(f"Nodes: {G.number_of_nodes():,}   Edges: {G.number_of_edges():,}")
print(f"Connected: {nx.is_connected(G)}   Self-loops: {nx.number_of_selfloops(G)}")
print(f"Mean degree: {2 * G.number_of_edges() / G.number_of_nodes():.1f}   density: {nx.density(G):.2e}")
assert set(G.nodes) == set(users.index) and nx.is_connected(G)
assert dict(G.degree()) == users.degree.to_dict(), "degrees must match the users table of step 1"

## 2b. Louvain from scratch

**Modularity** measures how much more densely the communities are connected inside than expected in a random graph with the same degrees:

$$Q=\sum_{C}\left[\frac{\Sigma_{in}(C)}{2m}-\gamma\left(\frac{\Sigma_{tot}(C)}{2m}\right)^2\right]$$

where $m$ is the number of edges, $\Sigma_{in}(C)$ is the sum of the degrees' internal part (every internal edge counted twice), $\Sigma_{tot}(C)$ is the sum of the degrees of the nodes in $C$, and $\gamma$ is the **resolution** ($\gamma=1$ is the standard definition from the course).

**Louvain** repeats two phases until modularity stops improving:
1. **Local moving.** Visit the nodes in random order. Remove node $i$ from its community and put it in the neighbouring community (or back in its own) that gives the largest gain
$$\Delta Q \propto k_{i,C}-\gamma\,\frac{\Sigma_{tot}(C)\,k_i}{2m},$$
where $k_{i,C}$ is the weight of the links from $i$ to $C$ and $k_i$ is the degree of $i$. Repeat until no node moves.
2. **Aggregation.** Collapse every community into one super-node. Links between communities are summed, and the links inside a community become a self-loop. Go back to phase 1 on the smaller graph.

The result is a hierarchy of partitions. We keep the last (coarsest) level. The algorithm is greedy and the visiting order is random, so different seeds can give different partitions. Section 2d measures how much.

Implementation notes: the graph is stored as a list of dictionaries (`adj[i][j] = weight`). A self-loop `adj[c][c]` holds the weight of all links inside the community with each internal edge counted twice. This way the degree of a super-node is just the sum of its dictionary and $2m$ never changes. The code uses only the standard library.

**Connectivity fix.** Louvain has a known defect (Traag, Waltman & van Eck, 2019): when a node that bridges two parts of a community moves away, the community can be left **internally disconnected**, and later aggregation steps can never split it again. We therefore add a post-processing step, `split_disconnected`, which replaces every disconnected community by its connected components. This can only **raise** $Q$: the internal edges stay the same, while $\sum_C \Sigma_{tot}(C)^2$ gets smaller. `louvain_connected` = Louvain + this step, and it is what we use from section 2d on (section 2c validates plain Louvain).

In [ ]:
def modularity(G, labels, gamma=1.0):
    # Modularity of a partition {node: community}, straight from the definition (independent of Louvain).
    m2 = 2 * G.number_of_edges()
    inside, total = Counter(), Counter()
    for u, v in G.edges():
        if labels[u] == labels[v]:
            inside[labels[u]] += 2
    for u, d in G.degree():
        total[labels[u]] += d
    return sum(inside[c] / m2 - gamma * (total[c] / m2) ** 2 for c in total)


def to_adjacency(G, index):
    adj = [dict() for _ in range(len(index))]
    for u, v in G.edges():
        if u != v:
            adj[index[u]][index[v]] = 1.0
            adj[index[v]][index[u]] = 1.0
    return adj


def local_moving(adj, gamma, rng):
    # Phase 1: move single nodes to the neighbouring community with the best gain until no node moves.
    n = len(adj)
    k = [sum(a.values()) for a in adj]          # degree of each (super-)node, self-loop included
    m2 = sum(k)                                  # 2m
    comm = list(range(n))                        # every node starts alone
    tot = k[:]                                   # sum of degrees of each community
    order = list(range(n))
    moved_any, improved = False, True
    while improved:
        improved = False
        rng.shuffle(order)
        for i in order:
            ci = comm[i]
            w = defaultdict(float)               # weight from i to each neighbouring community
            for j, wij in adj[i].items():
                if j != i:
                    w[comm[j]] += wij
            tot[ci] -= k[i]                      # take i out of its community
            best, best_gain = ci, w.get(ci, 0.0) - gamma * tot[ci] * k[i] / m2
            for c, wc in w.items():
                gain = wc - gamma * tot[c] * k[i] / m2
                if gain > best_gain + 1e-12:     # strict improvement, so ties keep the node where it is
                    best, best_gain = c, gain
            tot[best] += k[i]
            if best != ci:
                comm[i] = best
                improved = moved_any = True
    return comm, moved_any


def aggregate(adj, comm):
    # Phase 2: one super-node per community; parallel links are summed, internal links become a self-loop.
    ids = {c: x for x, c in enumerate(sorted(set(comm)))}
    c2 = [ids[c] for c in comm]
    new = [defaultdict(float) for _ in ids]
    for i, a in enumerate(adj):
        for j, w in a.items():
            new[c2[i]][c2[j]] += w
    return [dict(d) for d in new], c2


def louvain(G, gamma=1.0, seed=0):
    # Returns (partition {node: community}, list of partitions after each aggregation level).
    rng = random.Random(seed)
    nodes = list(G.nodes())
    index = {u: i for i, u in enumerate(nodes)}
    adj = to_adjacency(G, index)
    membership = list(range(len(nodes)))         # original node -> current super-node
    levels = []
    while True:
        comm, moved = local_moving(adj, gamma, rng)
        if not moved:
            break
        adj, c2 = aggregate(adj, comm)
        membership = [c2[m] for m in membership]
        levels.append({u: membership[index[u]] for u in nodes})
    if not levels:                               # nothing moved: every node is its own community
        levels.append({u: index[u] for u in nodes})
    return levels[-1], levels


def n_communities(part):
    return len(set(part.values()))


def split_disconnected(G, part):
    # Replace every internally disconnected community by its connected components.
    # Returns (new partition, number of communities that had to be split). Never lowers Q.
    groups = defaultdict(list)
    for u, c in part.items():
        groups[c].append(u)
    new, n_split, next_id = {}, 0, 0
    for c in sorted(groups):
        comps = list(nx.connected_components(G.subgraph(groups[c])))
        n_split += len(comps) > 1
        for comp in sorted(comps, key=lambda s: (-len(s), min(s))):   # deterministic order
            for u in comp:
                new[u] = next_id
            next_id += 1
    return new, n_split


def louvain_connected(G, gamma=1.0, seed=0):
    part, _ = louvain(G, gamma=gamma, seed=seed)
    return split_disconnected(G, part)[0]

## 2c. Validating our implementation

We cannot trust our own code just because it runs, so we check it in three ways:
1. **Modularity formula:** our `modularity()` must agree with `networkx` on the same partition.
2. **Zachary's karate club:** a 34-node graph where the best known Louvain modularity is about 0.419.
3. **The real graph:** the modularity of our Louvain must be in the same range as the one from `networkx.community.louvain_communities`.

In [ ]:
# 1+2. karate club
K = nx.Graph(nx.karate_club_graph().edges())   # drop the edge weights that newer networkx versions attach
q_ours = [modularity(K, louvain(K, seed=s)[0]) for s in range(30)]
p_ref = nx.community.louvain_communities(K, seed=0)
ref_labels = {u: i for i, c in enumerate(p_ref) for u in c}
assert abs(modularity(K, ref_labels) - nx.community.modularity(K, p_ref)) < 1e-12, "modularity() disagrees with networkx"
print(f"Karate club: our Louvain Q over 30 seeds = {np.mean(q_ours):.4f} (max {np.max(q_ours):.4f}); best known ~ 0.4198")
print("modularity() agrees with networkx on the reference partition: OK")

# 3. real graph, a quick comparison on 5 seeds
t0 = time.perf_counter()
ours = [modularity(G, louvain(G, seed=s)[0]) for s in range(5)]
t_ours = (time.perf_counter() - t0) / 5
t0 = time.perf_counter()
ref = [nx.community.modularity(G, nx.community.louvain_communities(G, seed=s)) for s in range(5)]
t_ref = (time.perf_counter() - t0) / 5
print(f"\nPhiladelphia graph, 5 seeds each:")
print(f"  ours     : Q = {np.mean(ours):.4f} +- {np.std(ours):.4f}   ({t_ours:.1f} s per run)")
print(f"  networkx : Q = {np.mean(ref):.4f} +- {np.std(ref):.4f}   ({t_ref:.1f} s per run)")

> **What we found.**
> - `modularity()` agrees with `networkx` on the same partition (once the edge weights that `networkx` 3.x attaches to the karate-club graph are dropped).
> - On the karate club our Louvain reaches a mean $Q$ = 0.4165 and a maximum of **0.4198, the best known value**.
> - On the Philadelphia graph our implementation gives $Q$ = 0.3756 ± 0.0078 against 0.3758 ± 0.0063 for `networkx`: the same quality, so we can use our own code for everything below.

## 2d. Many seeds: modularity and stability

Louvain is randomised, so a single run is not enough to make a claim. We run it with **`N_SEEDS` different seeds** and report:
- the modularity $Q$ and the number of communities of each run;
- the **stability**: how similar the partitions of two different runs are, measured with the Adjusted Rand Index (ARI) and the Normalised Mutual Information (NMI). If two runs disagree a lot, the community structure is not well defined and the later comparison with the taste communities would be built on sand.

Every run goes through the connectivity fix of section 2b, and we also count how often Louvain actually left a community disconnected.

We keep the run with the highest modularity as **the** friend partition. All 20 partitions are saved as well (section 2h), so that step 5 can repeat its comparison on each of them.

In [ ]:
runs = []
t0 = time.perf_counter()
for s in range(N_SEEDS):
    part_raw, levels = louvain(G, gamma=RESOLUTION, seed=s)
    part, n_split = split_disconnected(G, part_raw)
    runs.append({"seed": s, "Q": modularity(G, part), "Q_before_fix": modularity(G, part_raw),
                 "n_comm": n_communities(part), "split_communities": n_split,
                 "n_levels": len(levels), "part": part, "levels": levels})
print(f"{N_SEEDS} runs in {time.perf_counter() - t0:.0f} s")

runs_df = pd.DataFrame(runs).drop(columns=["part", "levels"]).set_index("seed")
print(runs_df.describe().loc[["mean", "std", "min", "max"]].round(4).to_string())
gain = runs_df.Q - runs_df.Q_before_fix
assert (gain >= -1e-12).all(), "splitting disconnected communities must never lower Q"
print(f"\nConnectivity fix: {(runs_df.split_communities > 0).sum()} of {N_SEEDS} runs had at least one disconnected community "
      f"({runs_df.split_communities.sum()} in total); Q gain at most {gain.max():.5f}")

order = list(G.nodes())
labels_mat = [[r["part"][u] for u in order] for r in runs]
ari = np.array([[adjusted_rand_score(a, b) for b in labels_mat] for a in labels_mat])
nmi = np.array([[normalized_mutual_info_score(a, b) for b in labels_mat] for a in labels_mat])
iu = np.triu_indices(N_SEEDS, k=1)
print(f"\nPairwise similarity between runs ({len(iu[0])} pairs):")
print(f"  ARI: mean {ari[iu].mean():.3f}   min {ari[iu].min():.3f}   max {ari[iu].max():.3f}")
print(f"  NMI: mean {nmi[iu].mean():.3f}   min {nmi[iu].min():.3f}   max {nmi[iu].max():.3f}")

best = max(runs, key=lambda r: r["Q"])
print(f"\nBest run: seed {best['seed']}  Q = {best['Q']:.4f}  communities = {best['n_comm']}")
print("Modularity after each aggregation level of the best run (before the fix):",
      [round(modularity(G, lv), 4) for lv in best["levels"]],
      "| communities:", [n_communities(lv) for lv in best["levels"]])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
axes[0].hist(runs_df.Q, bins=10, color=BLUE, edgecolor="white", linewidth=0.5)
axes[0].axvline(best["Q"], color=ORANGE, lw=2, label=f"chosen (Q = {best['Q']:.3f})")
axes[0].set(title="Modularity over seeds", xlabel="Q", ylabel="Runs")
axes[0].legend(frameon=False)
axes[1].bar(runs_df.index, runs_df.n_comm, color=BLUE)
axes[1].set(title="Communities per run", xlabel="Seed", ylabel="Communities")
im = axes[2].imshow(ari, cmap="Blues", vmin=ari[iu].min(), vmax=1)
axes[2].set(title="ARI between runs", xlabel="Seed", ylabel="Seed")
axes[2].grid(False)
fig.colorbar(im, ax=axes[2], fraction=0.046)
fig.tight_layout()
fig.savefig(FIGURES / "02_louvain_seeds.png")
plt.show()

> **What we found.**
> - **Modularity:** $Q$ = 0.372 ± 0.014 over 20 seeds (range 0.338–0.386). The chosen run (seed 9) has $Q$ = **0.3857** and **51 communities**. Values between 0.3 and 0.7 are usually read as a clear community structure, so the friendship graph is clustered, but only moderately.
> - **The number of communities varies a lot** between runs (31 to 83, mean 53). Most of the extra communities are tiny (see 2f).
> - **Stability is only moderate:** two runs agree with ARI 0.54 on average (min 0.30, max 0.84) and NMI 0.52. The large-scale structure comes back every time, but the boundaries between the big communities move from run to run.
> - Louvain finds most of its modularity early: the first level already has $Q$ = 0.374 (529 communities), and the later aggregations only add 0.012.
> - **Louvain's connectivity defect is real here:** 15 of the 20 runs left at least one community internally disconnected (23 such communities in total). The fix splits them; it raises $Q$ by at most 0.00004, so it changes almost nothing in quality, but it guarantees that every community is a connected group of friends. The chosen run had no disconnected community.
>
> **Consequence for the comparison with the taste communities.** Picking the best-$Q$ run is a legitimate choice, but it is still one draw. In step 5, NMI/ARI against the taste partition should also be computed on the other runs, which are saved in `philadelphia_pa_friend_communities_all_seeds.parquet` (section 2h), so that we can say how much of the result is just Louvain's randomness.

## 2e. The resolution parameter

Modularity has a known **resolution limit**: with $\gamma=1$ it can merge small communities or, on some graphs, split large ones. The parameter $\gamma$ moves the scale: larger values give more and smaller communities. We run Louvain for several values and look at how the number of communities changes, and how good each partition is **as judged by the standard modularity** ($\gamma=1$), which is the only value that is comparable across rows.

We keep $\gamma=1$ unless the table shows a good reason to change: it is the definition used in the course, it makes our numbers comparable with the literature, and there is no ground truth that would tell us which scale is the "right" one.

In [ ]:
res_rows = []
t0 = time.perf_counter()
for g in RESOLUTIONS:
    for s in range(SEEDS_PER_RESOLUTION):
        part = louvain_connected(G, gamma=g, seed=100 + s)
        sizes = np.array(sorted(Counter(part.values()).values(), reverse=True))
        res_rows.append({"gamma": g, "seed": s, "n_comm": len(sizes), "Q_standard": modularity(G, part, 1.0),
                         "Q_gamma": modularity(G, part, g), "largest": sizes[0],
                         "communities_ge10": int((sizes >= 10).sum())})
print(f"{len(res_rows)} runs in {time.perf_counter() - t0:.0f} s")
res_df = pd.DataFrame(res_rows)
res_table = res_df.groupby("gamma").agg(n_comm=("n_comm", "mean"), communities_ge10=("communities_ge10", "mean"),
                                        largest=("largest", "mean"), Q_standard=("Q_standard", "mean"),
                                        Q_standard_sd=("Q_standard", "std"))
res_table.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.3))
axes[0].errorbar(res_table.index, res_table.Q_standard, yerr=res_table.Q_standard_sd, marker="o", color=BLUE, capsize=3)
axes[0].axvline(RESOLUTION, color=ORANGE, ls="--", label="used")
axes[0].set(title="Standard modularity vs resolution", xlabel="gamma", ylabel="Q (gamma = 1)")
axes[0].legend(frameon=False)
axes[1].plot(res_table.index, res_table.n_comm, marker="o", color=BLUE, label="all communities")
axes[1].plot(res_table.index, res_table.communities_ge10, marker="o", color=ORANGE, label="with >= 10 users")
axes[1].set(title="Number of communities vs resolution", xlabel="gamma", ylabel="Communities")
axes[1].legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / "02_resolution.png")
plt.show()

> **What we found.**
> - **$\gamma = 1$ gives the best standard modularity** (0.380 ± 0.004) and also the lowest spread between seeds of all the values tried.
> - **$\gamma = 0.5$ is clearly too small:** the largest community swallows 5,345 users (67%), $Q$ drops to 0.20 and the runs disagree wildly (sd 0.11). This is the resolution limit in action.
> - Larger $\gamma$ splits the graph into more communities of reasonable size: the number of communities with at least 10 users goes from 12 ($\gamma = 1$) to 38 ($\gamma = 2$), at the cost of $Q$ (0.318 at $\gamma = 2$).
>
> **Decision:** we keep $\gamma = 1$ (best $Q$, the course definition). If the later comparison needs more, smaller friend communities, $\gamma = 1.5$ or $2$ is the natural alternative to report.

## 2f. Analysing the final communities

From here on, **the partition is the best run of section 2d**. Communities are renumbered by size (0 = largest). For each community we compute:
- size, and **internal density** (internal edges over possible pairs);
- the **share of edge ends that stay inside** the community (a community that keeps most of its friendships inside is well separated);
- its **contribution to $Q$**;
- how many **hubs** it contains (top 1% of users by number of friends) and what share of its internal edges touch a hub: a high share means the community is held together by a few very connected people;
- profile of the members: mean number of fans, median year of joining Yelp, mean rating.

In [ ]:
part = best["part"]
size_rank = {c: r for r, (c, _) in enumerate(Counter(part.values()).most_common())}
part = {u: size_rank[c] for u, c in part.items()}        # renumber: 0 = largest community
assert abs(modularity(G, part) - best["Q"]) < 1e-9

U = users.copy()
U["community"] = pd.Series(part)
hub_cut = U.degree.quantile(HUB_QUANTILE)
U["is_hub"] = U.degree >= hub_cut
print(f"Hubs: users with >= {hub_cut:.0f} friends -> {U.is_hub.sum()} users ({U.is_hub.mean():.1%})")

E = edges.copy()
E["cu"], E["cv"] = E.u.map(part), E.v.map(part)
E["hub_edge"] = E.u.map(U.is_hub) | E.v.map(U.is_hub)
m = len(E)

rows = []
for c, grp in U.groupby("community"):
    n = len(grp)
    e_int = int(((E.cu == c) & (E.cv == c)).sum())
    e_cut = int(((E.cu == c) != (E.cv == c)).sum())     # edges with exactly one end inside
    tot = int(grp.degree.sum())
    rows.append({"community": c, "users": n, "internal_edges": e_int, "boundary_edges": e_cut,
                 "density": e_int / (n * (n - 1) / 2) if n > 1 else np.nan,
                 "ends_inside": 2 * e_int / tot if tot else np.nan,
                 "Q_contribution": 2 * e_int / (2 * m) - (tot / (2 * m)) ** 2,
                 "hubs": int(grp.is_hub.sum()),
                 "hub_edge_share": float(E.loc[(E.cu == c) & (E.cv == c), "hub_edge"].mean()) if e_int else np.nan,
                 "mean_fans": grp.fans.mean(), "median_join_year": grp.yelping_since.dt.year.median(),
                 "mean_stars": grp.mean_stars.mean()})
comm_df = pd.DataFrame(rows).set_index("community")
assert abs(comm_df.Q_contribution.sum() - best["Q"]) < 1e-9, "contributions must add up to Q"

sizes = comm_df.users
print(f"Communities: {len(comm_df)}   sizes: largest {sizes.max():,}, median {sizes.median():.0f}, smallest {sizes.min()}")
print(f"Communities with >= 50 users: {(sizes >= 50).sum()} (they hold {sizes[sizes >= 50].sum() / sizes.sum():.1%} of the users)")
print(f"Communities with < 10 users : {(sizes < 10).sum()} (they hold {sizes[sizes < 10].sum()} users)")
print(f"Edges inside communities: {(E.cu == E.cv).mean():.1%}")
comm_df.head(15).style.format({"users": "{:,}", "internal_edges": "{:,}", "boundary_edges": "{:,}", "density": "{:.3f}",
                               "ends_inside": "{:.1%}", "Q_contribution": "{:.4f}", "hub_edge_share": "{:.1%}",
                               "mean_fans": "{:.1f}", "median_join_year": "{:.0f}", "mean_stars": "{:.2f}"})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
top = comm_df.head(20)
axes[0].bar(top.index.astype(str), top.users, color=BLUE)
axes[0].set(title="Size of the 20 largest communities", xlabel="Community (0 = largest)", ylabel="Users")
axes[0].tick_params(axis="x", labelsize=7)
cum = sizes.sort_values(ascending=False).cumsum() / sizes.sum()
axes[1].plot(np.arange(1, len(cum) + 1), cum.values, color=BLUE)
axes[1].set(title="Share of users covered by the k largest communities", xlabel="k", ylabel="Cumulative share of users")
fig.tight_layout()
fig.savefig(FIGURES / "02_community_sizes.png")
plt.show()

> **What we found.**
> - **The partition is very unbalanced.** Of 51 communities, only 9 have at least 50 users, and they hold 97.2% of the users; the five largest alone hold 91% (2,460 / 1,667 / 1,607 / 818 / 661). The other 38 communities have fewer than 10 users each (132 users in total).
> - **The three largest communities carry most of the modularity**: 0.119 + 0.084 + 0.094 = 0.297 of the total 0.386.
> - **63.6% of all friendships are inside a community**, so a user's friends are mostly, but not overwhelmingly, in the same community. For the five largest communities the share of edge ends that stay inside is 55–69%.
> - **Hubs sit in the big communities.** All 81 hubs (≥ 179 friends) are in the 9 largest communities (24 in community 0, 38 in community 2, 14 in community 1, 5 in community 4, none in community 3). They are 1% of the users but **45% of all friendships touch a hub**; in communities 0 and 2 about half of the internal edges do. Community 3 has no hub at all, so it is not held together by one.
> - **The communities differ in who is in them.** Community 4 has a mean of 113 fans per user, against 2–13 elsewhere: it looks like a group of very popular reviewers. Communities 0 and 1 differ by cohort (median join year 2014 against 2009); the next subsection shows how strong this effect is. Communities 5–9 have almost no fans (about 2 per user) and no hubs.
> - Internal density looks low (0.005 for the largest community) simply because the communities are large and the graph is sparse (overall density 0.002), so it should not be read as "weakly connected".

### Do the communities reflect *when* users were active?

The members' join years above already differ between communities. This matters for the whole project: two users who were active in the same years review restaurants that **existed in those years** (40% of the restaurants are now closed, see step 1). Their restaurant sets can then overlap without any shared taste. If the friend communities are largely "periods of activity", part of any friend–taste overlap found in step 5 could be explained by time alone.

For every user we take the **median year of their restaurant reviews**, and for each community with at least 50 users we compare the distribution of that year and the share of its reviews that are about restaurants closed today. We then measure how much the community label tells us about the period of activity (NMI between community and period, the period split into four bins), against the same NMI with shuffled community labels.

In [ ]:
reviews = pd.read_parquet(P + "reviews.parquet")
biz = pd.read_parquet(P + "businesses.parquet").set_index("business_id")
reviews["community"] = reviews.user_id.map(part)
reviews["closed_now"] = reviews.business_id.map(biz.is_open) == 0
U["median_review_year"] = reviews.groupby("user_id").date.median().dt.year

big_c = comm_df.index[comm_df.users >= 50]
B = U[U.community.isin(big_c)]
era = (B.groupby("community").median_review_year
         .describe()[["count", "25%", "50%", "75%"]]
         .rename(columns={"count": "users", "25%": "year_q25", "50%": "year_median", "75%": "year_q75"})
         .join(reviews[reviews.community.isin(big_c)].groupby("community").closed_now.mean().rename("reviews_of_closed")))

PERIOD_BINS = [0, 2010, 2013, 2016, 2100]           # <=2010, 2011-13, 2014-16, >=2017
period = pd.cut(B.median_review_year, PERIOD_BINS).astype(str)
nmi_era = normalized_mutual_info_score(B.community, period)
rng = np.random.default_rng(SEED)
nmi_shuffled = [normalized_mutual_info_score(rng.permutation(B.community.values), period) for _ in range(100)]
print(f"NMI(friend community, period of activity) = {nmi_era:.3f}   "
      f"shuffled labels: mean {np.mean(nmi_shuffled):.4f}, max {np.max(nmi_shuffled):.4f} (100 shuffles)")
era.style.format({"users": "{:,.0f}", "year_q25": "{:.0f}", "year_median": "{:.0f}", "year_q75": "{:.0f}",
                  "reviews_of_closed": "{:.1%}"})

In [ ]:
order_c = era.sort_values("year_median").index
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.boxplot([B.loc[B.community == c, "median_review_year"] for c in order_c], showfliers=False,
           patch_artist=True, boxprops=dict(facecolor=BLUE, alpha=0.6), medianprops=dict(color=ORANGE, lw=2))
ax.set_xticks(range(1, len(order_c) + 1), [f"{c}\n(n={int(era.users[c])})" for c in order_c], fontsize=8)
ax.set(title="Median review year of the members, per community (sorted by median)",
       xlabel="Community (0 = largest)", ylabel="Median review year")
ax.yaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
fig.tight_layout()
fig.savefig(FIGURES / "02_activity_period.png")
plt.show()

> **What we found.** **The friend communities are to a large extent periods of activity.**
> - The medians range from **2010** for community 1 (half of its members between 2009 and 2012) to **2018** for community 0 (2016–2019). Community 2 sits at 2013, communities 3–8 at 2014–2016.
> - The older the community, the more of its reviews are about restaurants that have since closed: **39%** for community 1 against **20%** for community 0 (29% over all reviews).
> - The community label says a lot about the period: NMI = **0.223**, against 0.001 (max 0.002) with shuffled labels.
>
> This is plausible (people who joined Yelp in the same years met at the same events and befriended each other) and it is not a flaw of the communities. **It is a confounder for step 5**: two friends active in the same years can only review the restaurants that were open then, so their restaurant sets overlap partly because of *time*, not *taste*. Step 5 should therefore check that a friend–taste overlap survives a control for the period, for example a permutation test that shuffles labels only **within** the same period, or a comparison of NMI(taste, friends) with NMI(taste, period).

## 2g. Robustness checks

Two questions decide whether the communities mean anything:

1. **Are they created by the hubs?** A handful of users with hundreds or thousands of friends can glue a graph together. We remove the hubs, run Louvain again on what is left (its largest connected component), and compare with the full partition on the users that remain.
2. **Is the modularity higher than chance?** Any sparse graph has some modularity. The fair reference is a random graph with **exactly the same degree of every user** (degree-preserving edge swaps), which keeps the hubs but destroys who is friends with whom. If our $Q$ is far above the random one, the community structure is real.

In [ ]:
# 1. remove the hubs
G_nh = G.subgraph([u for u in G if not U.is_hub[u]]).copy()
lcc = max(nx.connected_components(G_nh), key=len)
G_nh = G_nh.subgraph(lcc).copy()
print(f"Without hubs: {G_nh.number_of_nodes():,} users in the largest component "
      f"({G_nh.number_of_nodes() / (G.number_of_nodes() - U.is_hub.sum()):.1%} of the non-hubs), {G_nh.number_of_edges():,} edges")

nh_runs = [louvain_connected(G_nh, seed=s) for s in range(5)]
nh_best = max(nh_runs, key=lambda p: modularity(G_nh, p))
common = list(G_nh.nodes())
a = [part[u] for u in common]
nh_ari = [adjusted_rand_score(a, [p[u] for u in common]) for p in nh_runs]
nh_nmi = [normalized_mutual_info_score(a, [p[u] for u in common]) for p in nh_runs]
print(f"Q without hubs: {np.mean([modularity(G_nh, p) for p in nh_runs]):.4f}   communities: {n_communities(nh_best)}")
print(f"Agreement with the full partition on the remaining users: ARI {np.mean(nh_ari):.3f}, NMI {np.mean(nh_nmi):.3f}")
print(f"(for reference, agreement between two runs on the full graph: ARI {ari[iu].mean():.3f}, NMI {nmi[iu].mean():.3f})")

In [ ]:
# 2. degree-preserving random graphs
null_Q, null_n = [], []
t0 = time.perf_counter()
for s in range(N_NULL_GRAPHS):
    Gn = G.copy()
    nx.double_edge_swap(Gn, nswap=5 * G.number_of_edges(), max_tries=500 * G.number_of_edges(), seed=s)
    pn = louvain_connected(Gn, seed=s)
    null_Q.append(modularity(Gn, pn)); null_n.append(n_communities(pn))
    assert dict(Gn.degree()) == dict(G.degree())
print(f"{N_NULL_GRAPHS} random graphs in {time.perf_counter() - t0:.0f} s")
# Fair comparison: one Louvain run per random graph vs the AVERAGE run on the real graph (not the best of 20).
print(f"Random graphs with the same degrees: Q = {np.mean(null_Q):.4f} (values {np.round(null_Q, 4).tolist()}), "
      f"communities {null_n}")
print(f"Real friendship graph, mean run    : Q = {runs_df.Q.mean():.4f}  (best of {N_SEEDS} runs: {best['Q']:.4f})")

> **What we found.**
> - **Hubs blur the boundaries but do not create the communities.** Removing the 81 hubs leaves 7,113 users in one connected component (90.4% of the others). Louvain on that graph reaches $Q$ = 0.488 (35 communities), higher than with the hubs, which suggests that the hubs connect different communities. The new partition agrees with the full one on the remaining users with ARI 0.49 and NMI 0.48, about as much as two runs on the full graph agree with each other (0.54 / 0.52). The structure is therefore mostly the same, up to Louvain's own noise. ($Q$ values of two different graphs are not directly comparable, so this is only an indication.)
> - **The modularity is far above chance.** Random graphs with the same degree for every user still reach $Q$ = 0.171 (the heavy-tailed degrees alone create some apparent structure). The fair comparison is with an average run on the real graph, since each random graph also got a single run: **0.372, more than twice as much** (the best of 20 runs, 0.386, would flatter the real graph). The friendship communities are real and not an artifact of the degree distribution.

## 2h. Saving the partition

The output for the next steps is one row per user:

| File | One row per | Columns |
|---|---|---|
| `philadelphia_pa_friend_communities.parquet` | user in the LCC (7,947) | `user_id`, `friend_community` (integer, 0 = largest community) |
| `philadelphia_pa_friend_communities_all_seeds.parquet` | user in the LCC | `user_id`, `seed_0` … `seed_19`: the partition of every run of section 2d (after the connectivity fix) |
| `friend_communities_summary.json` | | parameters and the main numbers, to quote in the report |

Step 5 (NMI / ARI between friend and taste communities, permutation test) uses the first file for the main result and the second one to check that the result does not depend on Louvain's random seed. The community ids are **arbitrary labels**, and only the grouping matters.

In [ ]:
out = (pd.Series(part, name="friend_community").rename_axis("user_id").reset_index()
         .sort_values("user_id").reset_index(drop=True))
assert len(out) == users.shape[0] and out.user_id.is_unique and set(out.user_id) == set(users.index)
out.to_parquet(P + "friend_communities.parquet", index=False)

all_seeds = (pd.DataFrame({f"seed_{r['seed']}": pd.Series(r["part"]) for r in runs})
               .rename_axis("user_id").reset_index().sort_values("user_id").reset_index(drop=True))
assert len(all_seeds) == len(out) and (all_seeds.user_id.values == out.user_id.values).all()
assert adjusted_rand_score(all_seeds[f"seed_{best['seed']}"], out.friend_community) == 1.0, "chosen run must match"
all_seeds.to_parquet(P + "friend_communities_all_seeds.parquet", index=False)

summary = {
    "algorithm": "Louvain, implemented from scratch (this notebook)", "resolution": RESOLUTION,
    "n_seeds": N_SEEDS, "chosen_seed": int(best["seed"]),
    "graph": {"users": G.number_of_nodes(), "edges": G.number_of_edges()},
    "modularity": {"chosen": best["Q"], "mean_over_seeds": float(runs_df.Q.mean()), "sd_over_seeds": float(runs_df.Q.std()),
                   "min": float(runs_df.Q.min()), "max": float(runs_df.Q.max())},
    "communities": {"chosen": int(best["n_comm"]), "mean_over_seeds": float(runs_df.n_comm.mean()),
                    "largest": int(sizes.max()), "with_at_least_50_users": int((sizes >= 50).sum())},
    "stability_between_seeds": {"ARI_mean": float(ari[iu].mean()), "ARI_min": float(ari[iu].min()),
                                "NMI_mean": float(nmi[iu].mean()), "NMI_min": float(nmi[iu].min())},
    "without_hubs": {"hub_definition": f"top {1 - HUB_QUANTILE:.0%} by degree (>= {hub_cut:.0f} friends)",
                     "ARI_with_full_partition": float(np.mean(nh_ari)), "NMI_with_full_partition": float(np.mean(nh_nmi))},
    "random_degree_preserving_graphs": {"modularity_mean": float(np.mean(null_Q)), "modularity_values": null_Q,
                                        "compare_with": "modularity.mean_over_seeds (one run per graph on both sides)"},
    "connectivity_fix": {"runs_with_disconnected_communities": int((runs_df.split_communities > 0).sum()),
                         "disconnected_communities_total": int(runs_df.split_communities.sum()),
                         "max_Q_gain": float(gain.max())},
    "activity_period": {"NMI_community_vs_period": float(nmi_era), "NMI_shuffled_mean": float(np.mean(nmi_shuffled)),
                        "period_bins": "<=2010, 2011-2013, 2014-2016, >=2017 (median review year of the user)",
                        "communities_used": "communities with >= 50 users"},
    "share_of_edges_inside_communities": float((E.cu == E.cv).mean()),
    "files": ["philadelphia_pa_friend_communities.parquet", "philadelphia_pa_friend_communities_all_seeds.parquet"],
}
(DATA_PROCESSED / "friend_communities_summary.json").write_text(json.dumps(summary, indent=2))
print(f"{(P + 'friend_communities.parquet')} written: {len(out):,} rows")
print(f"{(P + 'friend_communities_all_seeds.parquet')} written: {all_seeds.shape[0]:,} rows x {all_seeds.shape[1] - 1} partitions")
print("friend_communities_summary.json written")
out.friend_community.value_counts().head(10)

### How later steps load the friend communities
```python
import pandas as pd
fc = pd.read_parquet("data/processed/philadelphia_pa_friend_communities.parquet")   # user_id, friend_community
friend_labels = fc.set_index("user_id")["friend_community"]
# e.g. NMI / ARI against a taste partition `taste_labels` (a Series indexed by user_id):
# from sklearn.metrics import normalized_mutual_info_score as nmi
# nmi(friend_labels.loc[common], taste_labels.loc[common])

# robustness: the same comparison on all 20 Louvain runs
all_seeds = pd.read_parquet("data/processed/philadelphia_pa_friend_communities_all_seeds.parquet").set_index("user_id")
# scores = [nmi(all_seeds.loc[common, col], taste_labels.loc[common]) for col in all_seeds.columns]
```

## Summary of step 2

**Result:** Louvain (our own implementation, validated against `networkx` and on the karate club) finds **51 friend communities** in the Philadelphia graph. A typical run reaches modularity **0.372 ± 0.014** (20 seeds; the chosen run 0.386), against 0.171 for random graphs with the same degrees. The communities are real, but they are unbalanced, only moderately stable between runs, and strongly tied to the period in which their members were active.

### Decision log

| # | Decision | Reason / evidence |
|---|---|---|
| 1 | Louvain **implemented from scratch** (local moving + aggregation) | Required for a technically challenging project; matches `networkx` on Q (0.3756 vs 0.3758) and reaches the best known 0.4198 on the karate club. |
| 2 | Resolution $\gamma = 1$ | Best standard modularity in the sweep (0.380); $\gamma = 0.5$ collapses into one giant community. |
| 3 | Final partition = best-$Q$ run of 20 seeds (seed 9) | Louvain is randomised; between-run agreement is only ARI 0.54, so the choice is reported, not hidden. |
| 4 | Keep the hubs | Removing them keeps the structure (ARI 0.49). Dropping them would remove 1% of the users and the 45% of all friendships that touch a hub. |
| 5 | Keep the 38 tiny communities (132 users) | Dropping them would lose users; they should be treated carefully (or merged) in step 5. |
| 6 | Split internally disconnected communities into their components | 15 of 20 Louvain runs produced at least one; splitting never lowers $Q$ (gain ≤ 0.00004) and makes every community a connected group of friends. |

### Notes for the next steps
- **Step 5 (comparison):** use `philadelphia_pa_friend_communities.parquet` for the main result. Because the partition is unbalanced (5 communities hold 91% of the users) and Louvain is not very stable (ARI 0.54), repeat NMI/ARI on the 20 runs in `philadelphia_pa_friend_communities_all_seeds.parquet` (and possibly with $\gamma = 1.5$).
- **Step 5, confounder:** friend communities are strongly tied to the period of activity (NMI 0.22 against 0.001 by chance, section 2f). The permutation test should shuffle labels within the same period, or the result should be compared with NMI(taste, period); otherwise a shared period could be mistaken for shared taste.
- **Limitations to report:** modularity of 0.37 is moderate; 36% of the friendships cross communities; selection bias of step 1 (only active, connected reviewers); Louvain leaves disconnected communities, which we fix afterwards (Leiden, not used here, avoids them by construction); friendships have no date, so we cannot tell whether friends met before or after reviewing.

---
*Step 3 — from `03_taste_communities.ipynb`.*

# Step 3 — Taste communities

**Project question:** *Are friend communities also taste communities?*
(DTU 02807 Computational Tools for Data Science, Yelp Open Dataset)

*Code: `src/taste.py` (functions), `src/lsh.py` (MinHash + LSH); this notebook runs them and explains the choices.*

**Question.** Do users who are friends on Yelp also share tastes? Step 2 built the **friend** partition of the 7,947 Philadelphia users. Here we build a **taste** partition of the *same* users, so step 5 can compare the two.

**Plan**
- **5.1** For each user, the set of things they like (two versions: restaurants and categories).
- **5.2** MinHash + LSH (from scratch) to find pairs of users with similar sets.
- **5.3** Taste-similarity graph → Louvain and Leiden communities.

## 0. Setup

All parameters are in this cell. Inputs are the files written by notebook 01 (`data/processed/philadelphia_pa_*.parquet`); nothing is recomputed from the raw Yelp JSON.

In [ ]:
import sys
from pathlib import Path

import json, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# make src/ importable whether Jupyter was started from the repo root or from notebooks/
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data" / "processed").exists())
sys.path.insert(0, str(ROOT / "src"))

import importlib, taste, lsh, louvain
importlib.reload(taste); importlib.reload(lsh); importlib.reload(louvain)

# --- parameters -----------------------------------------------------------
SEED = 42
POSITIVE_COL = "is_positive"     # "liked" = stars >= 3.5, same definition as step 1
GENERIC_MAX_FRAC = 0.80          # drop categories liked by more than 80 % of the users (see the sweep)
GENERIC_SWEEP = (0.3, 0.5, 0.7, 0.8, 0.9)
MAX_CATS_PER_BUSINESS = 20       # only Reading Terminal Market (35 categories; next highest 17) is left out of the category version
MIN_BUSINESSES_PER_CATEGORY = 5  # categories present in fewer restaurants describe one place, not a kind of food

# 5.2 MinHash + LSH
N_HASHES = 240                   # signature length (b * r)
K_NEIGHBOURS = 10                # taste graph: each user is linked to its 10 most similar users
ROWS_RANGE = {"business": (2, 5), "category": (2, 6)}   # r values tried in the (b, r) sweep
MIN_RECALL = 0.90                # the chosen (b, r) must find >= 90 % of the true nearest neighbours
MAX_BUCKET = 1000                # skip LSH buckets with more users than this (reported)
FALLBACK_BRUTE_FORCE = True      # if no (b, r) reaches MIN_RECALL, use the exact neighbours (reported)

# 5.3 communities (same settings as Mattia's friend communities, notebook 02)
N_SEEDS = 20
RESOLUTION = 1.0

RESULTS = ROOT / "data" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)
print("project root:", ROOT)

## 5.1 User sets

**What "liked" means.** A user likes a restaurant if `is_positive` is true (4 or 5 stars), the definition produced by step 1. We use the same definition as the rest of the project so that the friend partition, the taste partition and the association rules (step 4) talk about the same thing. The alternative, `stars_norm > 0` (above the user's own average), is kept as a possible robustness check.

**Two versions of each user's set**
1. **Restaurants**: the set of liked `business_id`s.
2. **Categories**: the set of categories of the liked restaurants.

Why two: friends may like the same restaurant simply because they went **together**, or because they were active in the **same years** (step 2 found that friend communities line up with the period of activity). Both inflate restaurant overlap between friends without meaning shared taste. Categories ("Thai", "Pizza") are much less affected.

We start from `categories_taste`, which step 1 already cleaned (it only removes `Restaurants`), and remove three kinds of categories that say nothing about taste:
- **Multi-purpose venues.** A few places (markets, food halls) are tagged with dozens of categories, many unrelated to food (*Sporting Goods, Department Stores, Dinner Theater…*). Liking one of them would add all those "tastes" at once; in a first run this created whole communities around a single venue. Restaurants with more than `MAX_CATS_PER_BUSINESS` = 20 categories are left out of the category version (they stay in the restaurant version). In practice this is only *Reading Terminal Market* (35 categories, twice the next one); a threshold of 10 would have removed 80 ordinary restaurants.
- **Rare categories**, present in fewer than `MIN_BUSINESSES_PER_CATEGORY` restaurants: they describe one place, not a kind of food.
- **Generic categories** that almost everybody likes (e.g. `Food`, `Nightlife`): they make every user look similar. We drop those liked by more than `GENERIC_MAX_FRAC` of the users.

In [ ]:
reviews, businesses, users = taste.load_inputs(ROOT)
print(f"{len(reviews):,} reviews | {len(businesses):,} restaurants | {len(users):,} users")
print(f"share of reviews that are positive: {reviews[POSITIVE_COL].mean():.1%}")

In [ ]:
# categories per restaurant: most have a handful, a few venues have dozens
n_cats = businesses["categories_taste"].apply(len)
print("categories per restaurant, quantiles 50/90/99 %:", n_cats.quantile([.5, .9, .99]).tolist(), " max:", n_cats.max())
display(businesses.assign(n_categories=n_cats).nlargest(10, "n_categories")[["name", "n_categories"]])

_, CAT_INFO = taste.clean_business_categories(businesses, MAX_CATS_PER_BUSINESS, MIN_BUSINESSES_PER_CATEGORY)
print(f"venues left out of the category version (> {MAX_CATS_PER_BUSINESS} categories): {len(CAT_INFO['venues_dropped'])}")
print(f"rare categories dropped (< {MIN_BUSINESSES_PER_CATEGORY} restaurants): {CAT_INFO['rare_categories_dropped']}, e.g. {CAT_INFO['rare_categories_examples'][:12]}")

### Choosing the generic-category threshold

How many categories are dropped, and how many users end up with an empty set, for different thresholds.

A common choice would be ~30 %, but on Yelp it is far too aggressive: users like on average 22 restaurants, so they "like" most common categories. At 30 % we would drop 57 categories, including real cuisines such as *Italian, Chinese, Mexican, Thai, Japanese, Vegan*, which are exactly what describes a user's taste. We choose **80 %**: it removes only the 7 categories nearly everybody likes (from *Food*, 97 %, to *Sandwiches*, 83 %), and there is a natural gap before the next one (*Italian*, 74 %).

In [ ]:
sweep = taste.generic_sweep(reviews, businesses, users, GENERIC_SWEEP, POSITIVE_COL,
                            MAX_CATS_PER_BUSINESS, MIN_BUSINESSES_PER_CATEGORY)
pd.set_option("display.max_colwidth", 120)
sweep

In [ ]:
sets_business = taste.build_user_sets(reviews, businesses, users, "business", POSITIVE_COL)
sets_category = taste.build_user_sets(reviews, businesses, users, "category", POSITIVE_COL, GENERIC_MAX_FRAC,
                                      MAX_CATS_PER_BUSINESS, MIN_BUSINESSES_PER_CATEGORY)

summary_51 = pd.DataFrame([sets_business.summary(), sets_category.summary()]).set_index("version")
summary_51

In [ ]:
print("Generic categories dropped (share of users who like them):")
for c, f in sets_category.dropped_items.items():
    print(f"  {c:<25} {f:.1%}")

print("\nMost frequent categories kept:")
taste.item_popularity(sets_category, top=10)

### Period of activity (for step 5)

Median review year per user, with the same bins as notebook 02. Not used to build the taste communities; saved so that step 5 can control for it.

In [ ]:
period = taste.user_activity_period(reviews)
period["period"].value_counts().sort_index()

### What we found (5.1)

- **7,941 of the 7,947 users** have a non-empty set in both versions. The 6 empty users have no 4–5-star review at all (not an effect of the filter); they are kept and flagged as isolated.
- **Restaurants:** 5,414 distinct restaurants; a user likes 22 on average (median 14, max 542).
- **Categories:** after leaving out Reading Terminal Market and 170 rare categories (present in < 5 restaurants), and dropping the 7 generic ones (*Food, Nightlife, Bars, American (New), Breakfast & Brunch, American (Traditional), Sandwiches*); see the summary table above for the final number of categories and set sizes.
- Category sets are **much denser** than restaurant sets, so Jaccard similarities are much higher in the category version (median of random pairs 0.19 vs 0.00).
- Limitation: the rare-category filter also removes a few genuine but very small cuisines (e.g. *Argentine, Armenian, Austrian*), which affects very few users.

## 5.2 MinHash + LSH: each user's most similar users

**Taste graph = k-nearest-neighbour graph.** Each user is linked to the `K_NEIGHBOURS` = 10 users whose sets are most similar (highest Jaccard), and the edge weight is that Jaccard.

*Why not a single similarity threshold?* We tried it first ("similar = more similar than 99.9 % of random pairs"): it left **85 % of the users isolated** in the category version and 20 % in the restaurant version. Jaccard depends strongly on set size, so users with few categories never reach a global threshold, and the communities would only describe the heaviest reviewers. With a kNN graph every user (with a non-empty set) has neighbours, whatever their absolute similarity.

**How LSH is used.** Finding each user's nearest neighbours exactly means comparing all 31.5 million pairs. LSH proposes **candidate pairs** instead:
1. **MinHash** turns each set into a signature of `N_HASHES` numbers; the share of equal positions in two signatures estimates their Jaccard.
2. **Banding** cuts the signature into `b` bands of `r` rows; two users are candidates if they are identical in at least one band. Probability $1-(1-s^r)^b$ (S-curve), threshold $t \approx (1/b)^{1/r}$.
3. For every candidate pair we compute the **exact** Jaccard and keep each user's top 10.

A pair is never a neighbour if its Jaccard is not above the **median Jaccard of random pairs** (`min_jaccard`): a "nearest neighbour" that is no more similar than a random user says nothing about taste.

Everything is implemented from scratch with NumPy in `src/lsh.py` ($h(x) = (a x + b) \bmod p$, $p = 2^{31}-1$). The exact kNN (**brute force** over all pairs) is the ground truth for recall and runtime.

In [ ]:
SETS = {"business": sets_business, "category": sets_category}
X, SIG = {}, {}
for v, S in SETS.items():
    X[v] = lsh.sets_to_csr(S.sets, len(S.item_ids))
    t = time.perf_counter()
    SIG[v] = lsh.minhash_signatures(S.sets, N_HASHES, SEED, len(S.item_ids))
    print(f"{v:<9} signatures {SIG[v].shape}  in {time.perf_counter() - t:.2f} s")

# sanity check: MinHash estimate vs exact Jaccard on 2,000 random pairs
rng = np.random.default_rng(SEED)
for v, S in SETS.items():
    p = rng.integers(0, len(S.sets), (2000, 2)); p = p[p[:, 0] != p[:, 1]]
    est = np.array([lsh.minhash_estimate(SIG[v], a, b) for a, b in p])
    exact = lsh.pair_jaccard(X[v], p)
    print(f"{v:<9} mean |MinHash estimate - exact Jaccard| = {np.abs(est - exact).mean():.4f}")

### Similarity by chance and of the nearest neighbours

Jaccard of 200,000 random pairs (what "no particular similarity" looks like) versus the exact Jaccard of each user's 10th nearest neighbour (the similarity LSH has to detect).

In [ ]:
rand_j = {v: lsh.random_pair_jaccard(X[v], 200_000, SEED) for v in SETS}
MIN_J = {v: float(np.median(rand_j[v])) for v in SETS}

TRUTH, T_BRUTE = {}, {}
for v in SETS:
    t = time.perf_counter()
    TRUTH[v] = lsh.brute_force_knn(X[v], K_NEIGHBOURS, MIN_J[v])
    T_BRUTE[v] = time.perf_counter() - t
KTH = {v: TRUTH[v].groupby("user")["jaccard"].min() for v in SETS}     # J of each user's k-th neighbour

pd.DataFrame({v: {"random pairs: median J (= min_jaccard)": MIN_J[v],
                  "random pairs: 99th pct": np.quantile(rand_j[v], .99),
                  "random pairs: share with J = 0": np.mean(rand_j[v] == 0),
                  f"{K_NEIGHBOURS}th neighbour: median J": KTH[v].median(),
                  f"{K_NEIGHBOURS}th neighbour: 10th pct J": KTH[v].quantile(.1),
                  "users with < k neighbours above min_jaccard": int((TRUTH[v].groupby('user').size() < K_NEIGHBOURS).sum()
                                                                    + len(SETS[v].sets) - TRUTH[v]['user'].nunique()),
                  "brute force kNN time (s)": T_BRUTE[v]} for v in SETS}).round(3)

### Choosing (b, r)

For each version we try several splits of the `N_HASHES` hash functions into `b` bands × `r` rows and record the number of candidates, the **recall** (share of the exact 10 nearest neighbours that LSH also finds) and the runtime. We keep the **cheapest configuration (fewest candidates) with recall ≥ 90 %**. If none reaches it, we use the exact neighbours from the brute force and say so.

In [ ]:
SWEEP, FOUND, CHOSEN = {}, {}, {}
for v, S in SETS.items():
    configs = lsh.band_configs(N_HASHES, ROWS_RANGE[v][1], ROWS_RANGE[v][0])
    SWEEP[v], FOUND[v] = lsh.knn_sweep(S.sets, len(S.item_ids), configs, K_NEIGHBOURS, TRUTH[v],
                                       MIN_J[v], SEED, MAX_BUCKET)
    best = lsh.choose_config(SWEEP[v], MIN_RECALL)
    reached = SWEEP[v].set_index(["bands", "rows"]).loc[best, "recall"] >= MIN_RECALL
    CHOSEN[v] = best if (reached or not FALLBACK_BRUTE_FORCE) else "brute force"

cols = ["bands", "rows", "t_approx", "n_candidates", "recall", "users_without_neighbour",
        "n_buckets_skipped", "max_bucket_seen", "time_total_s"]
for v in SETS:
    print(f"\n{v}: exact kNN (brute force) {T_BRUTE[v]:.1f} s -> chosen: {CHOSEN[v]}")
    display(SWEEP[v][cols])

In [ ]:
# S-curves of the chosen configurations, with the distribution of the 10th-neighbour Jaccard (figure for the report)
s = np.linspace(0, 1, 400)
colors = {"category": "#2a78d6", "business": "#eb6834"}
names = {"category": "categories", "business": "restaurants"}
fig, ax = plt.subplots(figsize=(6.5, 4))
for v in SETS:
    if CHOSEN[v] == "brute force":
        continue
    b, r = CHOSEN[v]
    ax.plot(s, lsh.s_curve(s, b, r), color=colors[v], lw=2)
    lo, med = KTH[v].quantile(.1), KTH[v].median()
    ax.axvspan(lo, KTH[v].quantile(.9), color=colors[v], alpha=0.12, lw=0)
    ax.annotate(f"{names[v]}: b={b}, r={r}", (med, lsh.s_curve(med, b, r)),
                xytext=(10, -14), textcoords="offset points", fontsize=9, color="0.25")
ax.set_xlabel("Jaccard similarity of a pair")
ax.set_ylabel("P(pair becomes a candidate)")
ax.set_title("LSH S-curves (shaded: 10th-90th pct of the 10th-neighbour Jaccard)", loc="left", fontsize=10)
ax.grid(alpha=0.25, lw=0.5); ax.spines[["top", "right"]].set_visible(False)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
fig.tight_layout(); fig.savefig(FIGURES / "03_lsh_s_curves.png", dpi=150)
plt.show()

### Save the taste-graph edges

Undirected kNN graph: edge u–v if v is among u's 10 nearest neighbours **or** u among v's. One file per version with the original Yelp ids, plus a JSON with the numbers of 5.1 and 5.2.

In [ ]:
EDGES = {}
for v, S in SETS.items():
    d = TRUTH[v] if CHOSEN[v] == "brute force" else FOUND[v][CHOSEN[v]]
    e = lsh.knn_edges(d)
    EDGES[v] = pd.DataFrame({"u": S.user_ids[e["u"].to_numpy()], "v": S.user_ids[e["v"].to_numpy()],
                             "jaccard": e["jaccard"].to_numpy()})
    EDGES[v].to_parquet(RESULTS / f"taste_edges_{v}.parquet", index=False)
    print(f"{v:<9} {len(EDGES[v]):,} edges saved")

def row(v):
    base = {"min_jaccard": round(MIN_J[v], 4), "kth_neighbour_median_j": round(float(KTH[v].median()), 4),
            "n_edges": len(EDGES[v]), "time_brute_force_s": round(T_BRUTE[v], 2)}
    if CHOSEN[v] == "brute force":
        best = SWEEP[v].loc[SWEEP[v]["recall"].idxmax()]
        return {**base, "method": "brute force (LSH recall too low)",
                "best_lsh": f"b={int(best['bands'])}, r={int(best['rows'])}: recall {best['recall']:.2f}"}
    r = SWEEP[v].set_index(["bands", "rows"]).loc[CHOSEN[v]]
    return {**base, "method": "LSH", "bands": CHOSEN[v][0], "rows": CHOSEN[v][1],
            "t_approx": float(r["t_approx"]), "n_candidates": int(r["n_candidates"]),
            "recall": float(r["recall"]), "time_lsh_s": float(r["time_total_s"]),
            "buckets_skipped": int(r["n_buckets_skipped"])}

summary = {"params": {"seed": SEED, "positive": POSITIVE_COL, "generic_max_frac": GENERIC_MAX_FRAC,
                      "max_cats_per_business": MAX_CATS_PER_BUSINESS,
                      "min_businesses_per_category": MIN_BUSINESSES_PER_CATEGORY,
                      "venues_dropped": CAT_INFO["venues_dropped"],
                      "rare_categories_dropped": CAT_INFO["rare_categories_dropped"],
                      "n_hashes": N_HASHES, "k_neighbours": K_NEIGHBOURS,
                      "min_recall": MIN_RECALL, "max_bucket": MAX_BUCKET},
           "user_sets": {v: S.summary() for v, S in SETS.items()},
           "lsh": {v: row(v) for v in SETS}}
(RESULTS / "taste_summary.json").write_text(json.dumps(summary, indent=2, default=str))
pd.DataFrame(summary["lsh"])

### What we found (5.2)

- **The nearest neighbours are meaningful.** A user's 10th most similar user has a median Jaccard of **0.43** (categories) and **0.10** (restaurants), against 0.19 and 0.00 for random pairs (74 % of random pairs share no liked restaurant).
- **A kNN graph instead of a global threshold.** A global threshold (top 0.1 % of random pairs) left 85 % of the users isolated in the category version, because Jaccard depends on set size. With 10 neighbours per user only the 6–7 users without a set are isolated and the graph is connected.
- **Categories: LSH works.** b = 80, r = 3 finds **99.5 %** of the exact neighbours. The sweep shows the trade-off: r = 2 → 23.9 M candidates (recall 1.00); r = 4 → 5.0 M (0.88); r = 6 → 0.7 M (0.38).
- **Restaurants: LSH is not enough.** The best configuration (b = 120, r = 2) only reaches recall **0.82**: the similarities we look for are low (J ≈ 0.10), and LSH is efficient only for high similarities. We use the exact neighbours from the brute force.
- **At this size brute force is faster** (2–4 s vs 74 s for LSH in the category version). Exact all-pairs comparison grows with n² (31.5 M pairs here) but still fits in a sparse matrix product; LSH grows roughly linearly and would pay off for the whole Yelp data (~2 M users), not for one city.

## 5.3 Taste-similarity graph and communities

**Graph.** One node per user (all 7,947, as in the friend graph), one edge per kNN pair from 5.2, **weight = Jaccard**: more similar pairs pull harder towards the same community. The few users without neighbours (empty set) stay as isolated nodes; each becomes its own community and is flagged (`isolated_*`), so step 5 can include or exclude them consistently.

**Algorithm.** We reuse **Mattia's Louvain written from scratch** (notebook 02, now in `src/louvain.py`), so both partitions are found with exactly the same method. The only change is that it can read edge weights (with `weight=None` it gives the same results as before). As in step 2 we run **20 seeds**, keep the run with the highest modularity, fix the communities that Louvain leaves internally disconnected (`split_disconnected`), and report stability between seeds.

In [ ]:
# check of the weighted extension: our modularity equals networkx's on a weighted graph
import networkx as nx
K = nx.karate_club_graph()                      # has edge weights
p, _ = louvain.louvain(K, seed=0, weight="weight")
groups = [{u for u in p if p[u] == c} for c in set(p.values())]
print("weighted modularity, ours:", round(louvain.modularity(K, p, weight="weight"), 4),
      " networkx:", round(nx.community.modularity(K, groups, weight="weight"), 4))

In [ ]:
ALL_USERS = users["user_id"].to_numpy()
G_TASTE, RUNS, PARTS, BEST = {}, {}, {}, {}
for v in SETS:
    G = taste.build_taste_graph(ALL_USERS, EDGES[v])
    G_TASTE[v] = G
    t = time.perf_counter()
    RUNS[v], PARTS[v] = taste.louvain_seeds(G, N_SEEDS, RESOLUTION, weight="weight")
    BEST[v] = int(RUNS[v].loc[RUNS[v]["modularity"].idxmax(), "seed"])
    n_iso = sum(1 for _, d in G.degree() if d == 0)
    n_comp = nx.number_connected_components(G.subgraph([u for u, d in G.degree() if d > 0]))
    print(f"{v:<9} {G.number_of_nodes():,} users, {G.number_of_edges():,} edges, {n_iso} isolated, "
          f"{n_comp} connected components (without isolated) | {N_SEEDS} Louvain runs in {time.perf_counter() - t:.0f} s")

In [ ]:
# diagnostic: are the isolated users the ones with small sets? (degree vs set size)
rows = []
for v, S in SETS.items():
    deg = dict(G_TASTE[v].degree())
    d = pd.DataFrame({"size": S.sizes, "degree": [deg[u] for u in S.user_ids]})
    d["size group"] = pd.qcut(d["size"], 4, labels=["Q1 (smallest sets)", "Q2", "Q3", "Q4 (largest sets)"])
    g = d.groupby("size group", observed=True).agg(set_size=("size", "median"), mean_degree=("degree", "mean"),
                                                   isolated=("degree", lambda x: (x == 0).mean()))
    g["version"] = v
    rows.append(g.reset_index())
pd.concat(rows).set_index(["version", "size group"]).round(2)

In [ ]:
def describe_runs(v):
    r, best = RUNS[v], RUNS[v].set_index("seed").loc[BEST[v]]
    sizes = pd.Series(PARTS[v][BEST[v]]).value_counts()
    non_iso = [u for u, d in G_TASTE[v].degree() if d > 0]
    stab = taste.pairwise_stability(PARTS[v], non_iso)
    return {"modularity (best run)": round(best["modularity"], 4),
            "modularity (mean ± sd, 20 seeds)": f"{r['modularity'].mean():.3f} ± {r['modularity'].std():.3f}",
            "best seed": BEST[v],
            "communities (best run)": int(best["n_communities"]),
            "communities with >= 50 users": int(best["n_communities_ge_50"]),
            "share of users in 5 largest": f"{best['share_users_in_5_largest']:.0%}",
            "5 largest sizes": " / ".join(str(x) for x in sizes.iloc[:5]),
            "disconnected communities fixed (all runs)": int(r["n_disconnected_fixed"].sum()),
            "runs with a disconnected community": int((r["n_disconnected_fixed"] > 0).sum()),
            "stability ARI mean (min)": f"{stab['ARI_mean']:.2f} ({stab['ARI_min']:.2f})",
            "stability NMI mean (min)": f"{stab['NMI_mean']:.2f} ({stab['NMI_min']:.2f})"}

COMM_SUMMARY = pd.DataFrame({v: describe_runs(v) for v in SETS})
COMM_SUMMARY

### Null model: are these communities real?

Same check as in step 2: modularity of Louvain on random graphs with the **same degrees** (and the same edge weights, shuffled). If the real modularity is much higher, the communities are not an artefact of the degree distribution.

In [ ]:
null_q = {}
for v in SETS:
    G = G_TASTE[v]
    qs = []
    for s in range(3):
        R = nx.Graph(nx.configuration_model([d for _, d in G.degree()], seed=s))
        R.remove_edges_from(nx.selfloop_edges(R))
        R = nx.relabel_nodes(R, dict(enumerate(G.nodes())))
        w = np.random.default_rng(s).permutation(EDGES[v]["jaccard"].to_numpy())
        for (a, b), x in zip(R.edges(), w):
            R[a][b]["weight"] = x
        p, _ = louvain.louvain(R, RESOLUTION, seed=s, weight="weight")
        qs.append(louvain.modularity(R, p, RESOLUTION, "weight"))
    null_q[v] = np.mean(qs)
    print(f"{v:<9} modularity: real {RUNS[v]['modularity'].max():.3f}   random graphs with same degrees {null_q[v]:.3f}")

### What characterises each taste community?

For the category version, the categories that are most **over-represented** in each large community (liked by ≥ 25 % of its members, ranked by lift = share in the community / share among all users). This is a sanity check that the communities mean something, and gives names for the report.

In [ ]:
pd.set_option("display.max_colwidth", 200)
taste.community_profile(sets_category, PARTS["category"][BEST["category"]], min_size=50)

In [ ]:
# same description for the restaurant-based communities (described with their members' categories)
taste.community_profile(sets_category, PARTS["business"][BEST["business"]], min_size=50)

In [ ]:
# figure: community sizes of the chosen partitions
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
for ax, v in zip(axes, ["category", "business"]):
    sizes = pd.Series(PARTS[v][BEST[v]]).value_counts().to_numpy()
    top = sizes[:15]
    ax.bar(range(len(top)), top, color=colors[v], width=0.8)
    ax.set_title(f"{names[v]}: {len(sizes)} communities", loc="left", fontsize=10)
    ax.set_xlabel("community (by size)")
    ax.set_xticks(range(0, len(top), 2))
    ax.grid(axis="y", alpha=0.25, lw=0.5); ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("users")
fig.suptitle("Taste communities: 15 largest", x=0.01, ha="left")
fig.tight_layout(); fig.savefig(FIGURES / "03_taste_community_sizes.png", dpi=150)
plt.show()

### Save the taste communities (contract with step 5)

`data/results/taste_communities.parquet`, one row per user of `philadelphia_pa_users.parquet` (7,947):

| column | meaning |
|---|---|
| `user_id` | original Yelp id |
| `louvain_business`, `louvain_category` | community of the chosen run (0 = largest; labels are arbitrary, compare groupings only) |
| `isolated_business`, `isolated_category` | True if the user has no similar pair (singleton community) |
| `median_year`, `period` | user's median review year and period bin (same bins as notebook 02), to control for the period-of-activity confounder |

Plus `taste_communities_all_seeds_{business,category}.parquet` (columns `seed_0` … `seed_19`) for the robustness check over the 20 runs, like step 2.

In [ ]:
out = pd.DataFrame({"user_id": ALL_USERS})
for v in SETS:
    out[f"louvain_{v}"] = out["user_id"].map(PARTS[v][BEST[v]]).astype("int64")
    deg = dict(G_TASTE[v].degree())
    out[f"isolated_{v}"] = out["user_id"].map(lambda u: deg[u] == 0)
    seeds = pd.DataFrame({"user_id": ALL_USERS})
    for s, p in PARTS[v].items():
        seeds[f"seed_{s}"] = seeds["user_id"].map(p).astype("int64")
    seeds.to_parquet(RESULTS / f"taste_communities_all_seeds_{v}.parquet", index=False)
out = out.merge(period.reset_index(), on="user_id", how="left")
out["period"] = out["period"].astype(str)
out.to_parquet(RESULTS / "taste_communities.parquet", index=False)

summary["communities"] = {v: {**COMM_SUMMARY[v].to_dict(), "modularity_null": round(null_q[v], 4)} for v in SETS}
summary["params"].update({"n_seeds": N_SEEDS, "resolution": RESOLUTION})
(RESULTS / "taste_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(out.shape); out.head()

### What we found (5.3)

- **Real communities.** Modularity **0.489** (categories) and **0.447** (restaurants), against **0.201** and **0.229** for random graphs with the same degrees.
- **Size.** 17 communities (categories; 9 with ≥ 50 users, 5 largest = 80 % of users) and 18 (restaurants; 12 with ≥ 50 users, 69 %).
- **Interpretable.** Most category communities have a clear profile: Japanese & sushi; pizza, cheesesteaks & Italian; pubs, sports bars & live music; bubble tea, ramen & Taiwanese/Cantonese; tapas & Spanish. The restaurant-based communities are similar (e.g. Korean & dim sum, Indian & halal, vegan & kosher).
- **Moderate stability.** ARI between seeds 0.32 (categories) and 0.39 (restaurants), lower than for the friend communities (0.54). The 20 runs are saved so that step 5 can report the spread, not only the chosen run.
- **Louvain defect.** Internally disconnected communities appeared in only 1 of 40 runs (fixed afterwards).
- **Limitation.** Two category communities (366 and 54 users) are still largely built around a single multi-category venue (market / food hall); together ~5 % of the users.

---
*Step 4 — from `04_frequent_itemsets.ipynb`.*

# Step 4 — Frequent itemsets and association rules (A-priori vs. PCY)

**Project question:** *Are friend communities also taste communities?*
(DTU 02807 Computational Tools for Data Science, Yelp Open Dataset)

*Code: `src/itemsets.py` (A-priori, PCY), `src/rules.py` (rules, comparison, null model), `src/benchmark.py` (runtime and memory). Tests: `tests/`. This notebook runs them and explains the choices.*

**Question.** Steps 2 and 3 split the same 7,947 Philadelphia users into **friend** communities and two kinds of **taste** communities. Here every user becomes a **basket** of the things they liked, and we ask which items are liked together (association rules) inside each community, and whether friend communities share rules the way taste communities do.

**Plan**
- **4.1** Baskets: restaurants liked, and categories of restaurants liked.
- **4.2** A-priori and PCY from scratch, and a check that they give identical results.
- **4.3** A-priori vs. PCY: runtime, memory, candidate pairs.
- **4.4** Association rules per community, for 3 community systems × 2 basket types.
- **4.5** Comparison of the rules across systems, against a null model, with the activity-period confounder.
- **4.6** Robustness over the 20 saved Louvain seeds.
- **4.7** Saved outputs.

## 0. Setup

All parameters are in this cell. Inputs are the files written by notebooks 01–03; nothing is recomputed from the raw Yelp JSON and no community is re-detected.

In [ ]:
import sys, json, subprocess, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data" / "processed").exists())
sys.path.insert(0, str(ROOT / "src"))

import importlib, taste, itemsets as I, rules as R, benchmark as BM
for m in (taste, I, R, BM):
    importlib.reload(m)

# --- parameters -----------------------------------------------------------
SEED = 42
# baskets = step-3 liked sets, with exactly notebook 03's category filter
GENERIC_MAX_FRAC = 0.80
MAX_CATS_PER_BUSINESS = 20
MIN_BUSINESSES_PER_CATEGORY = 5
MIN_BASKET_ITEMS = 2              # step 1: skip baskets that cannot contain a pair

# mining (one support / confidence per basket type, the same for every community system)
MIN_SUPPORT = {"business": 0.01, "category": 0.15}
MIN_CONFIDENCE = {"business": 0.25, "category": 0.50}
MIN_LIFT = 1.0                    # keep only positively associated rules
MAX_K = 3                         # largest itemset size
MIN_COUNT_FLOOR = 5               # min_count = max(ceil(min_support * n), 5)
MIN_COMMUNITY_SIZE = 100          # mining transactions needed to mine a community

# comparison
LIFT_RATIO_MIN = 1.5              # "community-specific" rule: lift >= 1.5 x population lift
TAG_SHARE_MAX = 0.5               # category rule flagged as a Yelp-tag artefact above this share
N_SHUFFLES = 20                   # null model: random partitions with the same sizes

# A-priori vs PCY
N_BUCKETS = 100_003               # prime; ~4x the number of distinct category pairs seen
BUCKET_SWEEP = (1_009, 10_007, 100_003, 1_000_003)
TIMING_REPS = 5

PROC, RESULTS, FIGURES = ROOT / "data" / "processed", ROOT / "data" / "results", ROOT / "figures"
RESULTS.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(exist_ok=True)
VERSIONS = ("business", "category")
print("project root:", ROOT)

## 4.1 Baskets

One basket per user. Items are either the **restaurants** the user liked or the **categories** of those restaurants. "Liked" is the stored `is_positive` column of step 1 (stars ≥ 3.5, i.e. 4 or 5 stars).

The baskets are exactly the liked sets of step 3: `taste.build_user_sets` with notebook 03's parameters. The category filter matters. Before it, *Food* is in 97% of the category baskets, *Nightlife* in 94% and *Bars* in 93%, and every rule would be built on them. The filter removes:
- categories liked by more than 80% of users;
- categories present in fewer than 5 restaurants;
- the one venue with more than 20 categories (Reading Terminal Market).

As in step 1, users with fewer than 2 items are not mining transactions, since they cannot contain a pair. They stay in the community files, and are simply absent when a community's baskets are collected.

In [ ]:
reviews, businesses, users = taste.load_inputs(ROOT)
friend = pd.read_parquet(PROC / "philadelphia_pa_friend_communities.parquet").set_index("user_id")["friend_community"]
taste_c = pd.read_parquet(RESULTS / "taste_communities.parquet").set_index("user_id")
ALL_USERS = users["user_id"]

# every community file must describe the same population
for name, idx in [("friend", friend.index), ("taste", taste_c.index)]:
    assert set(idx) == set(ALL_USERS) and idx.is_unique, f"{name} population differs"
assert reviews.duplicated(["user_id", "business_id"]).sum() == 0, "duplicate user-restaurant pairs"
assert reviews[["user_id", "business_id"]].notna().all().all(), "missing ids"

USER_SETS, BASKETS = {}, {}
for v in VERSIONS:
    USER_SETS[v] = taste.build_user_sets(reviews, businesses, users, v,
                                         generic_max_frac=GENERIC_MAX_FRAC,
                                         max_cats_per_business=MAX_CATS_PER_BUSINESS,
                                         min_businesses_per_category=MIN_BUSINESSES_PER_CATEGORY)
    BASKETS[v] = I.baskets_from_user_sets(USER_SETS[v], ALL_USERS, MIN_BASKET_ITEMS)

cat_long, cat_info = taste.clean_business_categories(businesses, MAX_CATS_PER_BUSINESS, MIN_BUSINESSES_PER_CATEGORY)
basket_summary = pd.DataFrame([{
    "basket_type": v, "users": len(ALL_USERS), "users_empty": len(USER_SETS[v].empty_users),
    "users_excluded_lt2": len(BASKETS[v].excluded_users), "mining_transactions": len(BASKETS[v].sets),
    "unique_items": BASKETS[v].n_items,
    "size_median": float(np.median([len(s) for s in BASKETS[v].sets])),
    "size_mean": round(float(np.mean([len(s) for s in BASKETS[v].sets])), 1),
    "size_p90": float(np.percentile([len(s) for s in BASKETS[v].sets], 90)),
    "size_max": int(max(len(s) for s in BASKETS[v].sets))} for v in VERSIONS])
display(basket_summary)
print("generic categories dropped:", USER_SETS["category"].dropped_items)
print("venues dropped from the category version:", cat_info["venues_dropped"])
print("rare categories dropped:", cat_info["rare_categories_dropped"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, v in zip(axes, VERSIONS):
    sizes = [len(s) for s in BASKETS[v].sets]
    ax.hist(sizes, bins=np.arange(0, np.percentile(sizes, 99) + 2), color="#4c72b0")
    ax.set(title=f"{v} baskets (n = {len(sizes):,})", xlabel="items per basket", ylabel="users")
fig.tight_layout(); fig.savefig(FIGURES / "04_basket_sizes.png", dpi=150); plt.show()

In [ ]:
# lookup tables: readable item names, and the business -> category table for the tag check
NAMES = {"business": businesses.set_index("business_id")["name"].reindex(BASKETS["business"].item_ids).to_numpy(),
         "category": BASKETS["category"].item_ids}
cat_index = pd.Series(np.arange(BASKETS["category"].n_items), index=BASKETS["category"].item_ids)
biz_rows = (cat_long[cat_long.category.isin(cat_index.index)]
            .assign(i=lambda d: cat_index.reindex(d.category).to_numpy())
            .groupby("business_id")["i"].apply(lambda s: np.sort(s.to_numpy())))
BIZ_INDEX = R.SupportIndex(list(biz_rows), BASKETS["category"].n_items)
POP = {v: R.SupportIndex(list(BASKETS[v].sets), BASKETS[v].n_items) for v in VERSIONS}
print(f"business-category table: {BIZ_INDEX.n:,} restaurants")

> **What we found.**
> - **Population.** All three community files contain the same 7,947 users, and there are no duplicate or missing user–restaurant pairs.
> - **Mining transactions.** 7,931 users have a restaurant basket with ≥ 2 items; 16 are excluded, 6 of them with no liked restaurant at all (the 16 of notebook 01). The category version keeps 7,938 users, because a single liked restaurant can carry several categories.
> - **Restaurant baskets are long-tailed and sparse.** 5,414 restaurants; median 14 per basket, 90th percentile 45, maximum 542.
> - **Category baskets are dense.** 196 categories after the filter; median 29 per basket. The 7 generic categories removed are the ones in 83–97% of baskets (*Food*, *Nightlife*, *Bars*, *American (New)*, …).

## 4.2 A-priori and PCY

Both are written from scratch in `src/itemsets.py`. Items are integer indices and itemsets are sorted tuples, so outputs are deterministic and can be compared exactly.

**A-priori** (course version)
1. Pass 1 counts every item.
2. Pass 2 counts pairs of frequent items in a **triangular array** of m(m−1)/2 counters, m = number of frequent items.
3. Sizes k ≥ 3 join frequent (k−1)-itemsets that share their first k−2 items, prune every candidate with an infrequent (k−1)-subset, and count the survivors against the transactions.

**PCY**
1. Pass 1 also hashes every pair of every basket into `N_BUCKETS` counters. The pair is packed into a 64-bit key (i << 32 | j), scrambled with the fixed MurmurHash3 finaliser, and reduced modulo `N_BUCKETS`. Python's salted `hash()` is not used, so results are reproducible.
   - *A first version used a linear hash (A·i + B·j) mod P. With A = 1,000,003 equal to one of the bucket counts, the i term vanished and 19,900 pairs fell into about 200 buckets. Results stayed correct, because counting is exact, but the filter stopped working. A regression test now checks how evenly pairs are spread.*
2. The bucket counts become a **bitmap** (1 bit per bucket, `np.packbits`) and the counter array is freed.
3. Pass 2 counts, in a **hash table** (dict), only pairs of frequent items whose bucket is frequent.
4. Sizes k ≥ 3 use the same A-priori extension as above. PCY's own contribution is the pair stage, so the comparison below is about pairs.

A collision can only make a bucket *look* frequent (a false positive), never hide a frequent pair. The bitmap is a filter, and exact supports are always counted afterwards.

**Support threshold.** `min_count = max(ceil(min_support × n), 5)` for each collection of n transactions.

In [ ]:
# unit tests: toy data with known answers, brute-force agreement, edge cases, forced collisions
out = subprocess.run([sys.executable, "-m", "pytest", "-q", str(ROOT / "tests")], capture_output=True, text=True, cwd=ROOT)
print(out.stdout.strip().splitlines()[-1])

In [ ]:
# the two algorithms on the full population: identical itemsets?
FULL = {}
for v in VERSIONS:
    T = list(BASKETS[v].sets)
    mc = I.min_support_count(MIN_SUPPORT[v], len(T), MIN_COUNT_FLOOR)
    a = I.apriori(T, BASKETS[v].n_items, mc, MAX_K)
    p = I.pcy(T, BASKETS[v].n_items, mc, N_BUCKETS, MAX_K)
    assert a.itemsets == p.itemsets, f"{v}: A-priori and PCY disagree"
    FULL[v] = a
    print(f"{v:<9} min_support {MIN_SUPPORT[v]:.0%} -> min_count {mc}: identical itemsets, by size {a.by_size()}")

> **What we found.**
> - **Tests.** All 37 tests pass. They cover:
>   - toy data compared with a brute-force miner;
>   - support exactly at the threshold;
>   - empty and singleton baskets;
>   - no frequent singletons;
>   - forced collisions (1 bucket);
>   - the triangular-array indexing;
>   - the bitmap;
>   - the spread of the hash;
>   - the "no false negatives" invariant;
>   - the rule metrics on a hand-computed example.
> - **Identical results.** On the full population, A-priori and PCY return identical itemsets: 579 / 293 / 7 frequent itemsets of size 1 / 2 / 3 for restaurants (1% support, min_count 80), and 80 / 1,390 / 9,568 for categories (15%, min_count 1,191).
> - **Restaurant itemsets stop at size 3.** Even the most popular restaurants co-occur rarely.

## 4.3 A-priori vs. PCY: runtime, memory, candidate pairs

Both algorithms run on **identical** inputs: the same transactions, item encoding and min_count. The benchmark covers the **pair stage** (`max_k = 2`), where the two algorithms differ. The k ≥ 3 stage is the same code for both and would only add identical time and memory. The inputs are the full population and the largest eligible community of each system, for both basket types.

How each quantity is measured:
- **Runtime:** median of `TIMING_REPS` runs, timed per pass with `time.perf_counter` inside the algorithms. Loading, plotting and rule generation are excluded.
- **Memory:** one extra run under `tracemalloc` (peak Python allocation; NumPy arrays are included because NumPy allocates through Python's tracer). Next to it, the theoretical size of each structure: course convention of 4 bytes per counter and 12 bytes per hash-table entry.
- **Candidate pairs:** reported as separate, defined counts (table below). One ambiguous "candidates" number would hide the trade-off.

`benchmark.benchmark` stops with an error if the two algorithms ever disagree.

In [ ]:
def largest_eligible(labels, v):
    for c in labels.value_counts().index:
        if len(BASKETS[v].subset(labels[labels == c].index)) >= MIN_COMMUNITY_SIZE:
            return c

SYSTEMS = {"friend": friend, "taste_business": taste_c["louvain_business"], "taste_category": taste_c["louvain_category"]}
bench_rows, cand_rows = [], []
for v in VERSIONS:
    colls = {"population": list(BASKETS[v].sets)}
    for s, lab in SYSTEMS.items():
        c = largest_eligible(lab, v)
        colls[f"{s} #{c}"] = BASKETS[v].subset(lab[lab == c].index)
    for name, T in colls.items():
        mc = I.min_support_count(MIN_SUPPORT[v], len(T), MIN_COUNT_FLOOR)
        rows, cand = BM.benchmark(T, BASKETS[v].n_items, mc, 2, N_BUCKETS, TIMING_REPS)
        bench_rows.append(rows.assign(basket_type=v, collection=name))
        cand_rows.append({"basket_type": v, "collection": name, "n_transactions": len(T), "min_count": mc, **cand})
BENCH = pd.concat(bench_rows, ignore_index=True)
CAND = pd.DataFrame(cand_rows)
cols = ["basket_type", "collection", "algorithm", "n_transactions", "min_count", "pass1_s", "pass2_pairs_s", "total_s", "total_s_min", "total_s_max", "peak_mem_mb"]
display(BENCH[cols].round(3))

In [ ]:
cand_cols = ["basket_type", "collection", "pairs_hashed_pass1", "nonempty_buckets", "frequent_buckets",
             "frequent_items", "apriori_candidate_pairs", "pcy_candidate_pairs", "removed_by_bitmap",
             "candidate_reduction_pct", "pcy_pairs_counted", "frequent_pairs"]
display(CAND[cand_cols].round(1))
mem_cols = ["basket_type", "collection", "triangular_bytes", "bucket_array_bytes", "bitmap_bytes", "hash_table_bytes_course"]
display((CAND[mem_cols].set_index(["basket_type", "collection"]) / 1024).round(1).add_suffix(" [KB]"))

**What the candidate-pair columns mean**

| Column | Definition |
|---|---|
| `pairs_hashed_pass1` | pair occurrences hashed by PCY in pass 1 (every pair of every basket) |
| `nonempty_buckets`, `frequent_buckets` | buckets with count > 0, and with count ≥ min_count |
| `apriori_candidate_pairs` | all pairs of frequent items, m(m−1)/2: what A-priori allocates and counts |
| `pcy_candidate_pairs` | pairs of frequent items whose bucket is frequent: PCY's candidate set |
| `removed_by_bitmap` | the difference; `candidate_reduction_pct` is it as a percentage |
| `pcy_pairs_counted` | candidates that actually occur in a basket, i.e. the hash-table entries |
| `frequent_pairs` | pairs with support ≥ min_count (identical for both algorithms) |

In [ ]:
pop_b = BENCH[BENCH.collection == "population"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
w = 0.38
for k, v in enumerate(VERSIONS):
    for j, algo in enumerate(("apriori", "pcy")):
        r = pop_b[(pop_b.basket_type == v) & (pop_b.algorithm == algo)].iloc[0]
        bottom = 0
        for part, col in [("pass1_s", "#4c72b0"), ("pass2_pairs_s", "#dd8452")]:
            axes[0].bar(k + (j - .5) * w, r[part], w, bottom=bottom, color=col,
                        label=part.replace("_s", "") if (k, j) == (0, 0) else None, edgecolor="white")
            bottom += r[part]
        axes[0].text(k + (j - .5) * w, bottom, algo, ha="center", va="bottom", fontsize=8)
        axes[1].bar(k + (j - .5) * w, r.peak_mem_mb, w, color=["#4c72b0", "#dd8452"][j], label=algo if k == 0 else None)
axes[0].set(xticks=range(2), xticklabels=VERSIONS, ylabel="seconds (median)", title="Runtime by pass (pair stage), full population"); axes[0].legend(fontsize=8)
axes[1].set(xticks=range(2), xticklabels=VERSIONS, ylabel="MB", title="Peak traced memory (pair stage), full population"); axes[1].legend()
cp = CAND[CAND.collection == "population"].set_index("basket_type")
x = np.arange(2)
for j, (col, lab) in enumerate([("apriori_candidate_pairs", "A-priori candidates"), ("pcy_candidate_pairs", "PCY candidates"),
                                ("pcy_pairs_counted", "PCY counted"), ("frequent_pairs", "frequent pairs")]):
    axes[2].bar(x + (j - 1.5) * 0.2, cp.loc[list(VERSIONS), col], 0.2, label=lab)
axes[2].set(xticks=x, xticklabels=VERSIONS, yscale="log", ylabel="pairs (log)", title="Candidate pairs, full population"); axes[2].legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGURES / "04_apriori_vs_pcy.png", dpi=150); plt.show()

In [ ]:
# effect of the number of buckets (pairs only, full population)
SWEEP = pd.concat([BM.bucket_sweep(list(BASKETS[v].sets), BASKETS[v].n_items,
                                   I.min_support_count(MIN_SUPPORT[v], len(BASKETS[v].sets), MIN_COUNT_FLOOR),
                                   BUCKET_SWEEP).assign(basket_type=v) for v in VERSIONS], ignore_index=True)
display(SWEEP.round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for v in VERSIONS:
    s = SWEEP[SWEEP.basket_type == v]
    axes[0].plot(s.n_buckets, s.candidate_reduction_pct, "o-", label=v)
    axes[1].plot(s.n_buckets, s.peak_mem_mb, "o-", label=v)
axes[0].set(xscale="log", xlabel="buckets", ylabel="% pairs removed by bitmap", title="Filtering vs. buckets")
axes[1].set(xscale="log", xlabel="buckets", ylabel="peak MB", title="PCY peak memory vs. buckets")
for a in axes: a.legend()
fig.tight_layout(); fig.savefig(FIGURES / "04_pcy_bucket_sweep.png", dpi=150); plt.show()

> **What we found** (pair stage, median of 5 runs).
>
> **Runtime: A-priori is faster everywhere, 3–5×.**
> - PCY's pass 1 hashes **every** pair of every basket: 4.5 M pair occurrences for restaurants, 6.4 M for categories. A-priori's pass 1 is a single count of items.
> - On the full population PCY's pass 1 alone takes 0.30 s (restaurants) and 0.40 s (categories), longer than A-priori's whole pair stage (0.12 s and 0.25 s).
> - Pass 2 is also slightly slower for PCY: a Python dict versus a NumPy array.
>
> **Candidate pairs: the bitmap filters well when frequent pairs are rare.**
> - Restaurants: 89% of the 167,331 pairs of frequent restaurants are removed, leaving 18,634 candidates for 293 frequent pairs (98–100% in the taste communities).
> - Categories: about 52% are removed. Of the 3,160 pairs of frequent categories, 1,390 *are* frequent, so no bitmap could remove more than 56%.
>
> **Memory, course model (4 bytes per counter, 12 per hash entry).**
> - The course's "1/3 rule": PCY saves memory in pass 2 only if it counts fewer than ⅓ of the pairs A-priori would count. Its table spends 12 bytes per pair against 4 for a triangular counter.
> - Restaurants, full population: rule met. PCY counts 18,405 pairs against 167,331 for A-priori. Pass 2 needs 12 KB (bitmap) + 216 KB (table) against a 654 KB triangular array, about 3× less.
> - Largest friend community: rule met, but only just. 72,048 of 256,686 pairs counted; 856 KB against 1,003 KB.
> - Categories: rule fails. With only 80 frequent categories, the triangular array is 12 KB, and PCY's table (18 KB) plus bitmap (12 KB) is larger. PCY's 391 KB bucket array in pass 1 is also far more than A-priori ever needs.
>
> **Memory, measured (tracemalloc).**
> - A-priori's peak is lower in every case, e.g. 2.6 MB vs. 10.0 MB for restaurants on the full population.
> - In Python, the peak comes from the per-chunk arrays of hashed pairs in pass 1, not from the counters. A-priori never builds pairs of infrequent items. The course argument about main memory (the bucket array fills memory that A-priori leaves idle in pass 1) assumes a memory-bound implementation, which this is not at this data size.
>
> **Number of buckets.**
> - Too few buckets make every bucket frequent and the filter useless: 1,009 and 10,007 buckets remove 0% of restaurant pairs.
> - 100,003 buckets remove 89%, and 1,000,003 remove 99.7% (480 candidates for 293 frequent pairs). But the bucket array then grows to 3.9 MB, against 654 KB for A-priori's triangular array.
> - For categories, the filter levels off at about 56%, the limit set by the truly frequent pairs.
>
> **Conclusion.** PCY pays off where it was designed to: many frequent items and few frequent pairs (restaurant baskets), with enough buckets. On dense category baskets, and at this data size in Python, A-priori is faster and smaller. We do **not** claim that PCY is faster.

## 4.4 Association rules per community

For a rule X → Y (X, Y non-empty and disjoint, X ∪ Y frequent):

$$\text{support} = \frac{s(X\cup Y)}{n},\qquad \text{confidence} = \frac{s(X\cup Y)}{s(X)},\qquad \text{lift} = \frac{\text{confidence}}{s(Y)/n}$$

All of these use the exact counts from mining. A rule describes **co-occurrence among liked items**, not causation.

**Two extra columns per rule**
- **`lift_pop`, `lift_ratio`**: the same rule's lift on the whole population, and lift in the community ÷ lift in the population. A large community with the same support fraction mostly reproduces global rules. A rule is **community-specific** when `lift_ratio ≥ 1.5`.
- **`same_business_share`** (category rules only): the share of restaurants tagged with all of X that are also tagged with all of Y. A high share means the rule restates how Yelp tags a single restaurant (*Sushi Bars → Japanese*), not two tastes of one user. Rules above 0.5 are flagged.

**Thresholds** (setup cell): one support and one confidence per basket type, the same for all three community systems.
- **Restaurant baskets are sparse.** The most-liked restaurant is in 15% of baskets. At 1% support the population has 293 frequent pairs; at 3% it has 1.
- **Category baskets are dense** (median 29 items after filtering). At 10% support the population already has 25k frequent triples.
- Confidence differs too: restaurant rules have a median confidence of about 0.2, category rules about 0.5.
- `MIN_COMMUNITY_SIZE = 100` mining transactions. Smaller communities, including the singleton communities of step 3's isolated users, are listed as excluded.

**Same algorithm everywhere.** Communities are mined with A-priori. Section 4.2 showed that PCY returns the same itemsets.

In [ ]:
SYSTEMS_ALL = {**SYSTEMS, "period": taste_c["period"].astype(str)}   # period: confounder check (4.5)

def params(v, max_k=MAX_K):
    return R.MiningParams(MIN_SUPPORT[v], MIN_CONFIDENCE[v], MIN_LIFT, max_k, MIN_COUNT_FLOOR,
                          MIN_COMMUNITY_SIZE, LIFT_RATIO_MIN)

SUMMARY, RULES = {}, {}
t0 = time.perf_counter()
for v in VERSIONS:
    for s, lab in SYSTEMS_ALL.items():
        SUMMARY[s, v], RULES[s, v] = R.mine_partition(BASKETS[v], lab, params(v), POP[v],
                                                      BIZ_INDEX if v == "category" else None)
print(f"mined 4 systems x 2 basket types in {time.perf_counter() - t0:.0f} s")

comm = pd.concat([d.assign(community_system=s, basket_type=v) for (s, v), d in SUMMARY.items()], ignore_index=True)
excl = (comm.groupby(["community_system", "basket_type"])
            .agg(communities=("community_id", "size"), eligible=("eligible", "sum"),
                 users_in_eligible=("n_assigned", lambda x: int(x[comm.loc[x.index, "eligible"]].sum())))
            .assign(excluded=lambda d: d.communities - d.eligible))
display(excl)

In [ ]:
show = ["community_system", "basket_type", "community_id", "n_assigned", "n_transactions", "n_items", "min_count",
        "frequent_k1", "frequent_k2", "frequent_k3", "rules_before_filter", "rules", "rules_specific"]
display(comm[comm.eligible & (comm.community_system != "period")][show])

In [ ]:
def readable(rules, v):
    n = NAMES[v]
    fmt = lambda t: " + ".join(str(n[i]) for i in t)
    return rules.assign(rule=[f"{fmt(a)} → {fmt(c)}" for a, c in zip(rules.antecedent, rules.consequent)])

def top_rules(s, v, k=3, specific=True):
    r = RULES[s, v]
    r = r[r.specific] if specific else r
    if v == "category":
        r = r[~(r.same_business_share >= TAG_SHARE_MAX)]
    r = r.sort_values(["community_id", "lift_ratio", "lift"], ascending=[True, False, False]).groupby("community_id").head(k)
    cols = ["community_id", "rule", "support", "confidence", "lift", "lift_pop", "lift_ratio"]
    return readable(r, v)[cols].round(3)

for v in VERSIONS:
    for s in SYSTEMS:
        print(f"\n=== {s} x {v}: top community-specific rules (max 3 per community{', tag rules removed' if v == 'category' else ''})")
        display(top_rules(s, v))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, v in zip(axes, VERSIONS):
    d = comm[comm.eligible & (comm.basket_type == v) & (comm.community_system != "period")]
    lab = d.community_system.str.replace("taste_", "t-") + " #" + d.community_id.astype(str)
    colors = d.community_system.map({"friend": "#4c72b0", "taste_business": "#dd8452", "taste_category": "#55a868"})
    ax.bar(range(len(d)), d.rules, color=colors)
    ax.bar(range(len(d)), d.rules_specific, color="black", alpha=.6, label="community-specific")
    ax.set(yscale="log", title=f"Rules per eligible community — {v} baskets", ylabel="rules (log)")
    ax.set_xticks(range(len(d))); ax.set_xticklabels(lab, rotation=90, fontsize=7); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGURES / "04_rules_per_community.png", dpi=150); plt.show()

In [ ]:
# same-business tag artefacts among category rules
tag = pd.DataFrame([{"community_system": s, "rules": len(RULES[s, "category"]),
                     "tag_flagged": int((RULES[s, "category"].same_business_share >= TAG_SHARE_MAX).sum()),
                     "pair_rules": len(R.pair_rules(RULES[s, "category"])),
                     "pair_tag_flagged": int((R.pair_rules(RULES[s, "category"]).same_business_share >= TAG_SHARE_MAX).sum())}
                    for s in SYSTEMS]).assign(share_flagged=lambda d: (d.tag_flagged / d.rules).round(3))
display(tag)
ex = readable(R.pair_rules(RULES["friend", "category"]), "category").drop_duplicates("rule")
display(ex.sort_values("same_business_share", ascending=False)[["rule", "same_business_share", "lift"]].head(8).round(2))

> **What we found.**
> - **Eligibility.** With ≥ 100 mining transactions, 7 of 51 friend communities, 11 of 17 restaurant-taste and 8 of 17 category-taste communities are mined. They hold 95%, 99.9% and 99% of the users. The excluded communities are the small friend groups and the singleton communities of step 3's isolated users.
> - **Rule counts are uneven, by construction.** One restaurant-taste community (#1) and one category-taste community (#4) are made of **heavy reviewers**, with median baskets of about 50 items against about 12 elsewhere. With a fixed support fraction almost every triple there is frequent, and those two communities alone produce most of the rules. This is why section 4.5 compares **pair rules**.
> - **Restaurant rules** describe destinations liked together: *Reading Terminal Market*, *Zahav*, *Dizengoff* and similar. In taste communities the specific rules are often local clusters, e.g. *Mike's BBQ ↔ South Philly Barbacoa* (lift 30 inside the community vs. 9.6 in the population) or *The Dandelion ↔ HipCityVeg*.
> - **Category rules** read as cuisines liked together (*Vegetarian → Vegan*, *Middle Eastern → Mediterranean*, *French → Wine Bars*), and these appear in **all 7** friend communities.
> - **Tag artefacts are few but real.** 2–3% of category rules have `same_business_share ≥ 0.5`. Pairs such as *Ramen → Japanese*, *Cantonese → Chinese* or *Falafel → Mediterranean* have a share of 1.0: every restaurant with the first tag also has the second. They are flagged and removed from the comparison.

## 4.5 Comparing rules across community systems

**Pair rules only.** Comparisons use rules with one item on each side.
- Rules from triples grow combinatorially in the communities of **heavy reviewers**. One restaurant-taste and one category-taste community have median baskets of about 50 items, against about 12 elsewhere. With a fixed support fraction almost every triple there is frequent, and those few communities would dominate every overlap number.
- Triple rules are still mined and saved (4.4).
- The direction of a rule is kept: A → B ≠ B → A.

**Measures, per system and basket type**
- `median_lift_ratio`, `share_lift_ratio_lt1`: does the partition **absorb** the population's associations? If a partition puts together users who like the same things, an association across the population is partly explained by community membership, so inside a community the lift drops towards 1 (`lift_ratio` < 1).
- `rules_specific`: rules with `lift_ratio ≥ 1.5`.
- `all_/specific_mean_pairwise_jaccard`: average Jaccard between the rule sets of two communities (low = different rules).
- `all_/specific_share_unique`: share of rules found in only one community.
- `jaccard_vs_*`: Jaccard between the union of a system's specific rules and another system's.

**Null model.** Each number is compared with `N_SHUFFLES` random partitions that have exactly the same community sizes (labels shuffled over users), as steps 2 and 3 did for modularity.

**The taste communities are a reference, not a finding.** They were built from these same liked sets, so they are *expected* to differ from random. The informative comparison is **friend vs. null, measured against how far taste goes**.

**Activity-period confounder.** Step 2 found that friend communities follow the users' period of activity (NMI ≈ 0.22). We therefore add the 4 period bins of `taste_communities.parquet` as a fourth "partition". If friend-specific restaurant rules are mostly period rules, they overlap with the period-specific rules.

Category rules flagged as Yelp-tag artefacts are excluded from the comparison.

In [ ]:
REAL, NULL = [], []
t0 = time.perf_counter()
for v in VERSIONS:
    tag_cut = TAG_SHARE_MAX if v == "category" else None
    spec = {s: R.union(R.rule_sets(R.pair_rules(RULES[s, v]), specific_only=True, exclude_tag=tag_cut)) for s in SYSTEMS_ALL}
    for s, lab in SYSTEMS_ALL.items():
        ref = {o: spec[o] for o in SYSTEMS_ALL if o != s}
        REAL.append({"basket_type": v, "community_system": s, **R.partition_measures(RULES[s, v], tag_cut, ref)})
        nm = R.null_model(BASKETS[v], lab, params(v, max_k=2), POP[v], N_SHUFFLES, SEED,
                          BIZ_INDEX if v == "category" else None, ref, tag_cut)
        NULL.append(nm.assign(basket_type=v, community_system=s))
REAL, NULL = pd.DataFrame(REAL), pd.concat(NULL, ignore_index=True)
print(f"null model: {N_SHUFFLES} shuffles x 4 systems x 2 basket types in {time.perf_counter() - t0:.0f} s")

In [ ]:
MEASURES = ["pair_rules", "median_lift_ratio", "share_lift_ratio_lt1", "rules_specific",
            "all_mean_pairwise_jaccard", "all_share_unique", "specific_share_unique"]

def vs_null(measures):
    rows = []
    for (v, s), g in NULL.groupby(["basket_type", "community_system"]):
        real = REAL[(REAL.basket_type == v) & (REAL.community_system == s)].iloc[0]
        for m in measures:
            if m not in g: continue
            mu, sd = g[m].mean(), g[m].std()
            rows.append({"basket_type": v, "community_system": s, "measure": m, "real": real[m],
                         "null_mean": mu, "null_sd": sd, "z": (real[m] - mu) / sd if sd > 0 else np.nan})
    return pd.DataFrame(rows)

CMP = vs_null(MEASURES)
order = list(SYSTEMS_ALL)
for v in VERSIONS:
    print(f"\n=== {v} baskets: real partition vs. {N_SHUFFLES} same-size random partitions")
    t = CMP[CMP.basket_type == v].pivot(index="measure", columns="community_system", values=["real", "null_mean", "z"])
    display(t.reindex(MEASURES).swaplevel(axis=1).reindex(columns=order, level=0).round(3))

In [ ]:
# overlap of community-specific rules between systems, with the null for comparison
J = vs_null([f"jaccard_vs_{o}" for o in SYSTEMS_ALL])
J = J.dropna(subset=["real"]).assign(other=lambda d: d.measure.str.replace("jaccard_vs_", ""))
display(J[J.community_system == "friend"][["basket_type", "community_system", "other", "real", "null_mean", "null_sd", "z"]].round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, v in zip(axes, VERSIONS):
    for k, s in enumerate(order):
        g = NULL[(NULL.basket_type == v) & (NULL.community_system == s)]["median_lift_ratio"]
        real = REAL[(REAL.basket_type == v) & (REAL.community_system == s)]["median_lift_ratio"].iloc[0]
        ax.scatter(np.full(len(g), k) + np.random.default_rng(k).uniform(-.12, .12, len(g)), g, s=12, color="grey", alpha=.6,
                   label="random partitions, same sizes" if k == 0 else None)
        ax.scatter([k], [real], s=90, marker="D", color="#c44e52", label="real partition" if k == 0 else None, zorder=3)
    ax.axhline(1, color="black", lw=.6, ls=":")
    ax.set(xticks=range(len(order)), xticklabels=order, ylabel="median lift_ratio of pair rules",
           title=f"{v} baskets: lift inside communities ÷ population lift")
    ax.tick_params(axis="x", rotation=20); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGURES / "04_lift_ratio_vs_null.png", dpi=150); plt.show()

In [ ]:
# rules shared by several friend communities, and specific friend rules that are also period rules
for v in VERSIONS:
    pr = R.pair_rules(RULES["friend", v])
    if v == "category":
        pr = pr[pr.same_business_share < TAG_SHARE_MAX]
    k = R.rule_keys(pr)
    n_comm = k.groupby(k).transform("size")
    shared = readable(pr[n_comm >= 3], v).groupby("rule").agg(communities=("community_id", "nunique"),
                                                             lift_min=("lift", "min"), lift_max=("lift", "max"))
    print(f"{v}: {len(set(k))} distinct friend pair rules; {int((k.value_counts() >= 3).sum())} appear in >= 3 friend communities")
    display(shared.sort_values(["communities", "lift_max"], ascending=False).head(5).round(2))

In [ ]:
# period composition of the eligible friend communities (confounder)
elig_f = comm[(comm.community_system == "friend") & comm.eligible & (comm.basket_type == "business")].community_id
mix = pd.crosstab(friend.reindex(taste_c.index), taste_c["period"], normalize="index").loc[elig_f]
display((mix * 100).round(0).astype(int).add_suffix(" %"))

> **What we found** (pair rules; z = (real − null mean) / null sd over 20 random partitions with the same sizes).
>
> **1. Taste communities absorb associations, as expected by construction.**
> - Inside taste communities, 95–98% of restaurant pair rules have a **lower** lift than in the population (median `lift_ratio` ≈ 0.6). Random partitions give 7–11% (median ≈ 1.3–1.4).
> - For categories the figures are 86–95% vs. ≈ 47%.
> - This is the circularity of section 1: the communities were built from these sets. It is the **reference** for how far a partition that tracks taste can go.
>
> **2. Friend communities absorb restaurant associations partially, and the period of activity does the same.**
> - Restaurants: 44.5% of friend pair rules have a lower lift inside the community, against 11% for random partitions (z ≈ 21). That is far from taste (95%) but well above chance.
> - The 4 **period bins alone** reach the same level: 44% (z ≈ 28).
> - Friend-specific restaurant rules overlap with **period-specific** rules about 4× more than chance (Jaccard 0.021 vs. 0.006, z ≈ 6). With taste-specific rules they overlap *less* than chance: 0 vs. 0.007 for restaurant-taste, 0.004 vs. 0.010 for category-taste.
> - The friend communities are themselves period-skewed. Friend community #0 is 72% ≥ 2017 users and #1 is 52% ≤ 2010.
> - **Interpretation:** much of what friend communities share at restaurant level is consistent with being active in the same years, when the same restaurants were open and popular. It is not evidence of shared taste. We did **not** control for period inside communities, so this is a strong confounder signal, not a decomposition.
>
> **3. Categories are not period-sensitive, and friend communities show only a weak taste signal there.**
> - For category rules the period partition is practically indistinguishable from random: 47.5% vs. 48.7% of rules below population lift, median `lift_ratio` 1.002 vs. 1.001. That supports the hypothesis that categories are insensitive to openings and closures.
> - Friend communities are still slightly above chance on categories: 55.6% vs. 46.5% (z ≈ 4.8), median `lift_ratio` 0.993 vs. 1.002. Taste communities reach 86–95%.
> - So friend communities carry a **small** category-level taste signal that the period does not explain. It is an order of magnitude weaker than the taste reference.
>
> **4. Rule sets: friend communities share most category rules.**
> - The average Jaccard between two friend communities' category rule sets is 0.43. Random partitions reach 0.80, taste communities 0.18–0.26.
> - 1,651 of 2,893 distinct friend category pair rules appear in ≥ 3 of the 7 friend communities, and the top ones (*Vegetarian → Vegan*, *French → Wine Bars*) in all 7.
> - Friend communities therefore have *somewhat* more distinct rules than chance, but far less than taste communities.
> - For restaurants all real partitions are more distinct than random (Jaccard 0.03 friend, 0.06 period, ≤ 0.02 taste, vs. ≈ 0.1–0.2 for random).
>
> **5. "Community-specific" counts must be read against the null.**
> - Random partitions produce *more* rules with `lift_ratio ≥ 1.5` than the real friend partition: 467 vs. 151 for restaurants.
> - In small random groups, the rules that pass the support threshold are the ones whose co-occurrence is high **by chance**. Their lift is inflated by sampling noise (winner's curse), and that inflates the specific-rule count.
> - The count of specific rules on its own is therefore **not** evidence that a community has its own taste. The lift-ratio distribution and the overlap with period and taste are more reliable.
> - For categories, friend communities have **no** specific pair rule at all.

## 4.6 Robustness over the 20 Louvain seeds

Steps 2 and 3 saved all 20 Louvain partitions. Friend stability was ARI 0.54; taste stability was 0.32–0.39. For every seed we rerun the pair-level measures of 4.5 (`MAX_K = 2`, which gives exactly the same pair rules) and the Jaccard between that seed's specific rules and the chosen partition's. Communities are matched **by rule content, never by label**.

In [ ]:
SEED_FILES = {"friend": (PROC / "philadelphia_pa_friend_communities_all_seeds.parquet", 9),
              "taste_business": (RESULTS / "taste_communities_all_seeds_business.parquet", None),
              "taste_category": (RESULTS / "taste_communities_all_seeds_category.parquet", None)}
taste_summary = json.loads((RESULTS / "taste_summary.json").read_text())
ROB = []
t0 = time.perf_counter()
for s, (path, chosen) in SEED_FILES.items():
    seeds = pd.read_parquet(path).set_index("user_id")
    if chosen is None:
        chosen = int(taste_summary["communities"][s.replace("taste_", "")]["best seed"])
    for v in VERSIONS:
        tag_cut = TAG_SHARE_MAX if v == "category" else None
        ref = R.union(R.rule_sets(R.pair_rules(RULES[s, v]), specific_only=True, exclude_tag=tag_cut))
        for col in seeds.columns:
            _, r = R.mine_partition(BASKETS[v], seeds[col], params(v, max_k=2), POP[v], BIZ_INDEX if v == "category" else None)
            m = R.partition_measures(r, tag_cut, {"chosen": ref})
            ROB.append({"community_system": s, "basket_type": v, "seed": int(col.split("_")[1]),
                        "is_chosen": int(col.split("_")[1]) == chosen, **m})
ROB = pd.DataFrame(ROB)
print(f"{len(ROB)} partitions mined in {time.perf_counter() - t0:.0f} s")
rob = (ROB[~ROB.is_chosen].groupby(["community_system", "basket_type"])
       [["median_lift_ratio", "share_lift_ratio_lt1", "rules_specific", "jaccard_vs_chosen"]]
       .agg(["mean", "min", "max"]).round(3))
display(rob)

> **What we found.**
> - **Partition-level measures are stable across seeds.** For restaurants, friend median `lift_ratio` is 0.99–1.11 over the other 19 seeds, against 0.57–0.70 for taste. The share of rules below population lift is 33–53% for friends and 86–99% for taste. The conclusions of 4.5 therefore do not depend on the chosen run.
> - **Individual specific rules are not stable.** The Jaccard between a seed's specific rules and the chosen partition's averages 0.13–0.20 for restaurants, and is mostly 0 for categories, where almost no specific rules exist.
> - We therefore report measures of the whole distribution, not lists of "the" rules of a community.
> - **Not covered:** the null model was run for the chosen partitions only, not for every seed.

## 4.7 Saved outputs

| File | One row per | Content |
|---|---|---|
| `data/results/step4_basket_summary.parquet` | basket type | users, exclusions, items, basket sizes |
| `data/results/step4_frequent_itemsets.parquet` | itemset (full population) | `basket_type`, `itemset_ids`, `itemset_names`, `size`, `support_count`, `support` |
| `data/results/step4_rules.parquet` | rule × community | `community_system`, `basket_type`, `community_id`, `antecedent_ids/names`, `consequent_ids/names`, `support_count`, `support`, `confidence`, `lift`, `confidence_pop`, `lift_pop`, `lift_ratio`, `specific`, `same_business_share`, `same_business_tag` |
| `data/results/step4_community_metrics.parquet` | community | sizes, threshold (fraction + count), itemsets by size, rule counts, eligibility + reason |
| `data/results/step4_benchmark.parquet` | algorithm × collection | runtime per pass (median), peak memory |
| `data/results/step4_candidates.parquet` | collection | candidate-pair counts and structure sizes |
| `data/results/step4_bucket_sweep.parquet` | bucket count | PCY filtering vs. memory |
| `data/results/step4_comparison.parquet`, `step4_null.parquet` | measure / shuffle | 4.5 real vs. null |
| `data/results/step4_seed_robustness.parquet` | seed | 4.6 |
| `data/results/step4_summary.json` | — | parameters, environment, headline numbers |

In [ ]:
def ids_names(t, v):
    return [str(BASKETS[v].item_ids[i]) for i in t], [str(NAMES[v][i]) for i in t]

def export_rules(r, v):
    a = [ids_names(t, v) for t in r.antecedent]; c = [ids_names(t, v) for t in r.consequent]
    out = r.drop(columns=["antecedent", "consequent"]).assign(
        antecedent_ids=[x[0] for x in a], antecedent_names=[x[1] for x in a],
        consequent_ids=[x[0] for x in c], consequent_names=[x[1] for x in c], basket_type=v)
    if "same_business_share" in out:
        out = out.assign(same_business_tag=out.same_business_share >= TAG_SHARE_MAX)
    return out

rules_out = pd.concat([export_rules(r, v).assign(community_system=s, community_id=lambda d: d.community_id.astype(str))
                       for (s, v), r in RULES.items() if not r.empty], ignore_index=True)
itemsets_out = pd.concat([pd.DataFrame([{"basket_type": v, "itemset_ids": ids_names(k, v)[0], "itemset_names": ids_names(k, v)[1],
                                         "size": len(k), "support_count": c, "support": c / FULL[v].n_transactions,
                                         "min_count": FULL[v].min_count} for k, c in FULL[v].itemsets.items()]) for v in VERSIONS],
                         ignore_index=True)
CAND_OUT = CAND.drop(columns=["candidates_k", "itemsets_by_size"]).assign(
    itemsets_by_size=CAND.itemsets_by_size.map(json.dumps), candidates_k=CAND.candidates_k.map(json.dumps))
outputs = {"step4_basket_summary": basket_summary, "step4_frequent_itemsets": itemsets_out, "step4_rules": rules_out,
           "step4_community_metrics": comm.assign(community_id=comm.community_id.astype(str)),
           "step4_benchmark": BENCH, "step4_candidates": CAND_OUT, "step4_bucket_sweep": SWEEP,
           "step4_comparison": pd.concat([CMP, J.drop(columns="other")], ignore_index=True), "step4_null": NULL,
           "step4_seed_robustness": ROB}
for name, df in outputs.items():
    df.to_parquet(RESULTS / f"{name}.parquet", index=False)
    print(f"{name}.parquet: {len(df):,} rows")

summary = {
    "params": {"seed": SEED, "positive": "is_positive (stars >= 3.5)", "generic_max_frac": GENERIC_MAX_FRAC,
               "max_cats_per_business": MAX_CATS_PER_BUSINESS, "min_businesses_per_category": MIN_BUSINESSES_PER_CATEGORY,
               "min_basket_items": MIN_BASKET_ITEMS, "min_support": MIN_SUPPORT, "min_confidence": MIN_CONFIDENCE,
               "min_lift": MIN_LIFT, "max_k": MAX_K, "min_count_rule": f"max(ceil(min_support * n), {MIN_COUNT_FLOOR})",
               "min_community_size": MIN_COMMUNITY_SIZE, "lift_ratio_min": LIFT_RATIO_MIN, "tag_share_max": TAG_SHARE_MAX,
               "n_shuffles": N_SHUFFLES, "n_buckets": N_BUCKETS, "timing_reps": TIMING_REPS,
               "comparison_rules": "pair rules (1 item -> 1 item)"},
    "environment": BM.environment(),
    "baskets": basket_summary.to_dict("records"),
    "full_population_itemsets": {v: FULL[v].by_size() for v in VERSIONS},
    "communities": excl.reset_index().to_dict("records"),
    "benchmark_population": BENCH[BENCH.collection == "population"][["basket_type", "algorithm", "total_s", "pass1_s", "pass2_pairs_s", "peak_mem_mb"]].to_dict("records"),
    "comparison_real_vs_null": CMP.round(4).to_dict("records"),
}
(RESULTS / "step4_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("step4_summary.json written")

> **Summary of step 4**
> - **A-priori and PCY**, both written from scratch, return identical frequent itemsets on every collection. See 4.3 for the runtime, memory and candidate-pair trade-off.
> - **Taste communities** absorb the population's associations almost completely (95–98% of restaurant rules, 86–95% of category rules have lower lift inside the community). This is expected, because they were built from the same liked sets.
> - **Friend communities**, at restaurant level, absorb associations about as much as the activity-period bins do (44% vs. 44%). Their specific rules overlap with period rules, not with taste rules. At category level, where the period has no effect, friend communities show a weak but significant taste signal (56% vs. 47% for random).
>
> **Limitations**
> - **Period confounding** was diagnosed, not removed. Stratifying by period inside communities would be the next step.
> - **Community imbalance and heavy-reviewer communities** make rule counts uneven, which is why comparisons use pair rules.
> - **Results depend on the thresholds**: support 1% / 15%, confidence 0.25 / 0.5, lift-ratio cut-off 1.5. No support-threshold sensitivity sweep was run.
> - **The null model** shuffles labels over users. It keeps community sizes but not the basket-size mix, e.g. it does not reproduce heavy-reviewer communities.
> - **Rules describe co-occurrence of liked items, not causation.** Similar rules in friend communities do not show that friendship causes shared preferences.

---
*Step 5 — from `05_comparison.ipynb`.*

# Step 5 — Comparison of friend and taste communities

**Project question:** *Are friend communities also taste communities?*
(DTU 02807 Computational Tools for Data Science, Yelp Open Dataset)

*Code: `src/comparison.py` (functions, tested in `tests/test_comparison.py`); this notebook runs them and explains the choices.*

**Question.** Are friend communities also taste communities? Step 2 built a **friend** partition and step 3 a **taste** partition of the *same* 7,947 Philadelphia users. Here we measure how much they agree and whether the agreement is more than chance.

**The trap this step has to avoid.** Step 2 found that friend communities line up with *when* their members were active on Yelp (NMI 0.22 against 0.001 by chance). Two friends active in the same years can only review restaurants that were open then, so they look alike for reasons that have nothing to do with taste. Every test below is therefore run twice: once ignoring the period, and once **controlling for it** (shuffling only among users of the same period bin). The second one is the one that counts.

**Plan**
- **1** Inputs and sanity check.
- **2** The measures: NMI, ARI (and AMI).
- **3** Partition level: agreement between the friend and the taste partition (chosen runs, and the 20 × 20 saved runs of each); the noise ceiling (how much two runs of the *same* method agree, the most one can ask of a comparison); baselines against the period bins; a permutation test (is the agreement above what the period alone produces?); sensitivity to the isolated users and the tiny communities.
- **4** Pair level: do friend pairs share a taste community / liked items more than other pairs?
- **5** Restaurants vs categories: does the answer depend on how taste is represented?
- **6** Figures.
- **7** Save.
- **8** What we found.

## How to run

1. Raw Yelp files in `data/raw/` (original names), then run, in this order: `01_data_cleaning.ipynb` → `02_friend_communities.ipynb` → `03_taste_communities.ipynb`. `data/` is not in git, so every team member regenerates these files locally.
2. Run this notebook (*Run All*) from the repository root or from `notebooks/`. About 6–10 minutes on a laptop. The same analysis can be run from the command line with `python src/comparison.py`.
3. Inputs read (nothing is recomputed from the raw data): `data/processed/philadelphia_pa_friend_communities*.parquet`, `philadelphia_pa_friend_edges.parquet`, `philadelphia_pa_{reviews,businesses,users}.parquet` and `data/results/taste_communities*.parquet`.
4. Outputs: `data/results/comparison_*.json/csv` (not in git) and `figures/05_*.png`.

Dependencies: `pip install numpy pandas scipy scikit-learn matplotlib pyarrow networkx` (see `requirements.txt`).

## 0. Setup

All parameters are in this cell.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# make src/ importable whether Jupyter was started from the repo root or from notebooks/
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data" / "processed").exists())
sys.path.insert(0, str(ROOT / "src"))
import comparison as C

# --- parameters -----------------------------------------------------------
SEED = 42
N_PERM_MAIN = 10_000    # permutations for the chosen pair of partitions (smallest possible p = 1/10,001)
N_PERM_GRID = 1_000     # permutations for each of the 20 x 20 grid cells
N_PERM_OTHER = 1_000    # period baselines, sensitivity variants, friend-pair share
N_PERM_PAIRS = 1_000    # friend-pair Jaccard test (the report uses P = 1000)
STRICT = True           # stop if the regenerated partitions differ from the numbers in the report

FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 70)
print("project root:", ROOT)

## 1. Inputs and sanity check

`load_inputs` reads the files of steps 1–3 and puts them on the same users. **Sanity check:** each team member regenerates the data, so a different library version could change a Louvain run without anyone noticing. `check_against_report` compares what we loaded with what the report says (7,947 users; 51 friend communities; 17 category and 17 restaurant taste communities in the chosen runs, which include the singleton communities of the users isolated in the taste graph). With `STRICT = True` the notebook stops if anything differs.

In [ ]:
inp = C.load_inputs(ROOT)
sanity = C.check_against_report(inp, STRICT)
print("found:", sanity["found"])
print("matches the report:", sanity["ok"])

print(f"\n{len(inp.edge_u):,} friend pairs; {inp.friend_seeds.shape[1]} friend runs; "
      f"{ {v: inp.taste_seeds[v].shape[1] for v in C.VERSIONS} } taste runs")
pd.DataFrame({"users": np.bincount(inp.period)}, index=inp.period_labels).T

In [ ]:
# community sizes of the chosen partitions (largest first)
def sizes(x, k=8):
    s = pd.Series(x).value_counts().to_numpy()
    return f"{len(s)} communities; largest {', '.join(map(str, s[:k]))} ..."

print("friends:          ", sizes(inp.friend_main))
for v in C.VERSIONS:
    print(f"taste, {v:<9}:", sizes(inp.taste_main[v]), f"| isolated users: {int(inp.isolated[v].sum())}")

## 2. The measures

* **NMI** (normalised mutual information): how much knowing a user's friend community tells us about their taste community; 0 = nothing, 1 = identical partitions. Arithmetic-mean normalisation (the scikit-learn default). With many small clusters NMI is **inflated by chance**, which is why it is never read without its null.
* **ARI** (adjusted Rand index): fraction of user pairs on which the two partitions agree, corrected for chance; 0 = random, 1 = identical. Dominated by the big communities (5 friend communities hold 91 % of the users).
* **AMI**: NMI corrected for chance with the exact expected value; reported for the headline pairs only (it is too slow for thousands of permutations).
* Community labels are arbitrary: only the grouping matters, never the label numbers.

**Noise ceiling.** Louvain runs on different seeds agree only moderately (ARI 0.54 between friend runs, about 0.32 between taste runs). Two partitions cannot agree with each other more than each agrees with itself, so the run-to-run agreement is shown next to every result.

**Permutation test, controlling for the period.** The friend partition stays fixed; taste labels are shuffled **among users of the same period bin** (≤ 2010, 2011–13, 2014–16, ≥ 2017, the median year of each user's reviews). That keeps the link between taste and period but destroys the link between taste and friends, so the null distribution answers: *how much agreement does the period alone produce?* p = (1 + #{null ≥ observed}) / (1 + P). The **plain** shuffle (all users together) is reported too, only to show how much the control matters.

First, our NMI/ARI against scikit-learn on the chosen partitions (raises an error if they differ):

In [ ]:
for v in C.VERSIONS:
    view = C.make_view(inp, v)
    print(v, {k: round(x, 6) for k, x in C.verify_against_sklearn(view.f_main, view.t_main).items()})

## 3. Partition level: category taste (main) and restaurant taste

For each taste version `analyse_version` runs: the chosen pair with `N_PERM_MAIN` permutations, the 20 × 20 grid of saved runs (each cell with its own within-period null), the noise ceilings, the period baselines, the sensitivity variants and the friend-pair share. The category version is the main one (Cesi's recommendation: restaurant sets overlap partly because friends went together or were active in the same years).

Users with no similar pair in the taste graph (6–8 users) are singleton "communities" and are **excluded** from the main analysis; the sensitivity block shows the effect of including them.

In [ ]:
res_cat, tab_cat = C.analyse_version(inp, "category", N_PERM_MAIN, N_PERM_GRID, N_PERM_OTHER, SEED)

In [ ]:
res_bus, tab_bus = C.analyse_version(inp, "business", N_PERM_MAIN, N_PERM_GRID, N_PERM_OTHER, SEED)

In [ ]:
results = {"parameters": {"n_perm_main": N_PERM_MAIN, "n_perm_grid": N_PERM_GRID, "n_perm_other": N_PERM_OTHER,
                          "n_perm_pairs": N_PERM_PAIRS, "seed": SEED, "min_community": C.MIN_COMMUNITY,
                          "period_bins": inp.period_labels},
           "sanity": sanity, "versions": {"category": res_cat, "business": res_bus}}
tables = {"category": tab_cat, "business": tab_bus}

### 3.1 Chosen partitions against both null models

In [ ]:
C.main_table(results).round(4)

Read it as: *observed* against *chance, shuffled within period*. `excess` is the part of the agreement the period does not explain; `z` and `p` say whether it is distinguishable from chance. If the plain-shuffle row is much lower than the within-period row, the period is doing part of the work. *Inside period bins* compares friends and tastes only among users active in the same period.

### 3.2 Robustness over the 20 × 20 saved runs

In [ ]:
C.grid_table(results).round(4)

Each cell is one friend run against one taste run, with its own within-period null (same permutations for all cells). `share of cells with p<0.05` is the fraction of the 400 comparisons in which the result is not explained by the period.

### 3.3 Noise ceiling and period baselines (the table for the report)

In [ ]:
C.build_table_v(results)

The decisive comparison: **friend vs taste** against **taste vs period** and **friend vs period**. If the period explains the taste partition as well as the friendships do, "friends share tastes" cannot be told apart from "friends were active in the same years". The two ceilings bound what any comparison between runs can reach.

### 3.4 Sensitivity: isolated users and tiny communities

In [ ]:
C.sensitivity_table(results).round(4)

Steps 2 and 3 note that the partitions are unbalanced and have many tiny communities, which can inflate NMI. The variants include the isolated users (each a singleton community), merge every community with fewer than 10 users into one group, or drop their users.

## 4. Pair level: friend pairs against other pairs

The partition-level test depends on two noisy Louvain partitions. This block asks the question directly, without communities: **do the 67,622 friend pairs have more in common than other pairs of users?** Two statistics:

* the share of friend pairs whose two users are in the **same taste community** (all 20 taste runs);
* the **mean Jaccard similarity** of the two users' liked sets (and the share of pairs with at least one liked item in common).

Two null models: **random user pairs** (as many random pairs as friend pairs; the version in the report) and **within period** (every user's taste data is swapped with a random user of the same period bin, the friend edges stay fixed, so each friend pair is compared with pairs active in the same periods).

The liked sets are rebuilt with `taste.build_user_sets` using the parameters notebook 03 saved in `taste_summary.json` (note that the function's own defaults differ from the notebook's: 0.30 vs 0.80 for the generic-category threshold and 10 vs 20 for the maximum number of categories per restaurant), and checked against the sizes notebook 03 reports. Only users with a non-empty set enter the Jaccard test.

In [ ]:
mats, set_info = C.build_taste_matrices(ROOT, inp.users, STRICT)
results["sets"] = set_info
results["jaccard"], pair_nulls = C.analyse_pairs(inp, mats, N_PERM_PAIRS, SEED + 6)
set_info

In [ ]:
C.pair_table(results).round(4)

## 5. Restaurant-based vs category-based taste partitions

Do the two representations of taste describe the same communities? (Chosen runs, users isolated in neither version.)

In [ ]:
results["taste_vs_taste"] = C.taste_vs_taste(inp)
results["taste_vs_taste"]

## 6. Figures

In [ ]:
C.plot_main_vs_null(results, tables, FIGURES / "05_main_vs_null.png")
plt.show()

In [ ]:
C.plot_agreement_levels(results, FIGURES / "05_agreement_levels.png")
plt.show()

In [ ]:
C.plot_pair_tests(results, pair_nulls, FIGURES / "05_pair_tests_report.png", column=True)   # one-column version for the report
plt.close()
C.plot_pair_tests(results, pair_nulls, FIGURES / "05_pair_tests.png")
plt.show()

## 7. Save

`data/results/comparison_summary.json` has every number of this notebook; the CSV files have the 20 × 20 grids and the null distributions. They are not in git: regenerate them by running this notebook.

In [ ]:
results["table_v"] = C.build_table_v(results).to_dict(orient="records")
written = C.save_results(ROOT, results, tables, pair_nulls)
print("\n".join(str(p.relative_to(ROOT)) for p in written))

## 8. What we found

*Numbers from `data/results/comparison_summary.json` (strict run of 2026-11-25). The restaurant taste partition has 19 communities (13 real + 6 isolated singletons), the count agreed by the team; the first committed run of notebook 03 had 18. The category version has 17 (9 real + 8 singletons).*

**Short answer: friends are measurably more alike in taste than other users, but friend communities are not taste communities.**

**1. Partition level: real but small agreement.** Chosen runs, isolated users excluded:

| | NMI | chance, within period | chance, plain shuffle | ARI | chance, within period | AMI |
|---|---|---|---|---|---|---|
| category taste | 0.042 | 0.021 | 0.011 | 0.013 | 0.005 | 0.031 |
| restaurant taste | 0.078 | 0.041 | 0.014 | 0.037 | 0.020 | 0.065 |

The observed agreement is about twice the within-period chance level, with p = 1/10,001 (the smallest possible) for both measures and both versions. Controlling for the period matters: it raises the chance NMI 2 to 3 times compared with the plain shuffle, so a test without the control would have overstated the effect.

**2. Robust across runs and variants.** Over the 20 × 20 saved runs, NMI is 0.042 [0.033, 0.053] (category) and 0.067 [0.040, 0.085] (restaurant), against within-period chance means of 0.022 and 0.035. 100 % of the 400 cells beat the within-period null for NMI in both versions and for restaurant ARI. Category ARI is the weakest result: 95.75 % of the cells have p < 0.05, and the worst cell has p = 0.975. In the sensitivity variants (isolated users included, tiny communities merged or dropped) NMI stays between 0.035 and 0.043 (category) and 0.068 and 0.079 (restaurant), always with p = 0.001.

**3. Small compared with the ceiling and with the period.** Two runs of the *same* method agree far more than friends and tastes do: friend vs friend runs NMI 0.522 (ARI 0.538), taste vs taste runs NMI 0.423 / 0.445 (ARI 0.315 / 0.383). Friend–taste agreement is between about a twentieth and a fifth of that, depending on the measure and version. Friend communities are organised much more by *when* members were active than by taste: friend vs period NMI 0.200 (ARI 0.196), against taste vs period NMI 0.028 (category) and 0.059 (restaurant). So the period does not explain the taste partition, but it explains the friend partition far better than taste does.

**4. Pair level: the clearest signal.** Chosen taste partitions and liked sets, friend pairs against same-period pairs (random-pair values in brackets):

| | friend pairs | same-period pairs | random pairs |
|---|---|---|---|
| same category taste community | 0.265 | 0.150 | (0.145) |
| same restaurant taste community | 0.211 | 0.101 | (0.089) |
| mean Jaccard, liked categories | 0.306 | 0.210 | (0.209) |
| mean Jaccard, liked restaurants | 0.023 | 0.009 | (0.008) |
| share with a liked restaurant in common | 0.684 | 0.284 | (0.265) |

All p = 1/1,001 (the smallest possible). Friend pairs share a taste community 1.8 (category) to 2.1 (restaurant) times as often as same-period pairs, and their liked restaurants overlap about 2.6 times more. The pair level and the partition level tell the same story: a consistent excess over chance, small in absolute terms at the community level.

**5. Restaurants vs categories.** The two taste partitions agree only moderately with each other (NMI 0.121, ARI 0.074, 7,939 users). The restaurant version shows a stronger friend–taste link everywhere (partition, grid, pairs). Friends overlap more on *which places* they like than on *what kind of food*. That fits going out together or living in the same neighbourhood as much as similar preferences.

**Why the method behaved like this.** Agreement between two partitions is capped by how noisy each one is (run-to-run ARI 0.54 for friends, 0.32 to 0.38 for taste), and the taste partition of the category version is dominated by a few large communities (9 real communities, the 5 largest holding 80 % of the users). A modest friend–taste effect at the level of individual pairs is therefore diluted at the level of communities. Low community overlap and friends being more similar than other pairs are not in contradiction: both are what this run shows.

**Limitations.** The period control is coarse (4 bins of the median review year) and does not remove neighbourhood, going to restaurants together, or friend influence: the data cannot separate influence from homophily. Grid cells share partitions, so the share of cells with p < 0.05 is a robustness indicator, not a multiple-testing p-value.